# TaniAdaptif - Complete Multicrop LoRA Pipeline

**Revision:** `2026-08-14-imagefolder-probe-v8`

Notebook orkestrator final untuk Google Colab GPU. Urutan eksekusi:
preparation -> smoke test 100 steps -> validasi smoke -> main training
1600 steps -> validasi main -> evaluasi otomatis base vs LoRA.

Input utama dibaca dari `/content/dataset_multicrop_v1.zip`. Drive hanya
menyimpan output, evaluasi, dan log persisten pada `MyDrive/skripsi_lora/`.
Notebook berhenti jika checksum, konfigurasi, artefak tahap sebelumnya, CUDA,
atau metadata tidak valid. Jangan menjalankan notebook ini pada laptop lokal.


In [ ]:
import csv
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

if "COLAB_RELEASE_TAG" not in os.environ:
    raise SystemExit("Pipeline diblokir: notebook ini hanya untuk Google Colab.")

PIPELINE_REVISION = "2026-08-14-imagefolder-probe-v8"
print("Pipeline revision:", PIPELINE_REVISION)

PACKAGE_PATH = Path("/content/dataset_multicrop_v1.zip")
UPLOAD_PARTS_DIR = Path("/content/dataset_multicrop_v1_upload_parts")
EXPECTED_SHA256 = "f6aa060658cf95e48996f10f0ffde200938d340dd15e6b60adace5bb5cecfb33"

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def assemble_runtime_package(
    package_path, parts_dir, expected_sha256, drive_package_path=None
):
    package_path = Path(package_path)
    parts_dir = Path(parts_dir)
    if package_path.is_file() and sha256_file(package_path) == expected_sha256:
        return package_path, "existing_zip"
    if package_path.exists() and not package_path.is_file():
        raise SystemExit(f"STOP: path dataset bukan file regular: {package_path}")

    if drive_package_path is not None:
        drive_package_path = Path(drive_package_path)
        if drive_package_path.is_file():
            drive_sha256 = sha256_file(drive_package_path)
            if drive_sha256 == expected_sha256:
                temporary_path = package_path.with_name(
                    f".{package_path.name}.from-drive"
                )
                try:
                    temporary_path.unlink(missing_ok=True)
                    with drive_package_path.open("rb") as source:
                        with temporary_path.open("wb") as target:
                            shutil.copyfileobj(source, target, length=1024 * 1024)
                    if sha256_file(temporary_path) != expected_sha256:
                        raise SystemExit(
                            "STOP: salinan ZIP dari Google Drive tidak valid."
                        )
                    os.replace(temporary_path, package_path)
                finally:
                    temporary_path.unlink(missing_ok=True)
                print("ZIP valid disalin dari Google Drive:", drive_package_path)
                return package_path, "google_drive"
            print(
                "WARNING: checksum ZIP Google Drive tidak cocok; "
                "mencoba folder potongan runtime."
            )

    manifest_path = parts_dir / "upload_manifest.json"
    if not manifest_path.is_file():
        raise SystemExit(
            "STOP: ZIP valid tidak ditemukan di Google Drive maupun runtime. "
            f"Folder potongan juga belum ditemukan di {parts_dir}."
        )
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError) as exc:
        raise SystemExit(f"STOP: manifest potongan upload tidak valid: {exc}") from exc

    parts = manifest.get("parts")
    expected_size = manifest.get("source_size_bytes")
    if (
        manifest.get("schema_version") != 1
        or manifest.get("source_filename") != package_path.name
        or manifest.get("source_sha256") != expected_sha256
        or not isinstance(expected_size, int)
        or expected_size <= 0
        or not isinstance(parts, list)
        or not parts
        or manifest.get("part_count") != len(parts)
    ):
        raise SystemExit("STOP: kontrak manifest potongan upload tidak valid.")

    validated_parts = []
    observed_names = set()
    for item in parts:
        name = item.get("name") if isinstance(item, dict) else None
        size_bytes = item.get("size_bytes") if isinstance(item, dict) else None
        part_sha256 = item.get("sha256") if isinstance(item, dict) else None
        if (
            not isinstance(name, str)
            or Path(name).name != name
            or name in observed_names
            or not isinstance(size_bytes, int)
            or size_bytes <= 0
            or not isinstance(part_sha256, str)
        ):
            raise SystemExit("STOP: entry manifest potongan upload tidak valid.")
        part_path = parts_dir / name
        if not part_path.is_file() or part_path.stat().st_size != size_bytes:
            raise SystemExit(f"STOP: potongan upload hilang/tidak lengkap: {part_path}")
        actual_part_sha256 = sha256_file(part_path)
        if actual_part_sha256 != part_sha256:
            raise SystemExit(
                f"STOP: checksum potongan upload tidak cocok: {name}; "
                f"expected={part_sha256}, actual={actual_part_sha256}"
            )
        observed_names.add(name)
        validated_parts.append(part_path)

    temporary_path = package_path.with_name(f".{package_path.name}.assembling")
    if temporary_path.exists() and not temporary_path.is_file():
        raise SystemExit(f"STOP: path assembly sementara bukan file: {temporary_path}")
    try:
        temporary_path.unlink(missing_ok=True)
        with temporary_path.open("wb") as target:
            for part_path in validated_parts:
                with part_path.open("rb") as source:
                    shutil.copyfileobj(source, target, length=1024 * 1024)
        actual_size = temporary_path.stat().st_size
        actual_sha256 = sha256_file(temporary_path)
        if actual_size != expected_size or actual_sha256 != expected_sha256:
            raise SystemExit(
                "STOP: hasil penyusunan ZIP tidak valid. "
                f"size={actual_size}/{expected_size}, "
                f"sha256={actual_sha256}/{expected_sha256}"
            )
        os.replace(temporary_path, package_path)
    finally:
        temporary_path.unlink(missing_ok=True)

    if not package_path.is_file() or sha256_file(package_path) != expected_sha256:
        raise SystemExit("STOP: validasi ZIP setelah atomic replace gagal.")
    print(f"ZIP berhasil disusun dari {len(validated_parts)} potongan:", package_path)
    return package_path, "assembled_parts"

from google.colab import drive

drive.mount("/content/drive")
DRIVE_PACKAGE_PATH = Path(
    "/content/drive/MyDrive/skripsi_lora/datasets/dataset_multicrop_v1.zip"
)
PACKAGE_PATH, PACKAGE_INPUT_MODE = assemble_runtime_package(
    PACKAGE_PATH, UPLOAD_PARTS_DIR, EXPECTED_SHA256, DRIVE_PACKAGE_PATH
)
print("Package input mode:", PACKAGE_INPUT_MODE)

actual_sha256 = sha256_file(PACKAGE_PATH)
if actual_sha256 != EXPECTED_SHA256:
    raise SystemExit(
        "STOP: checksum ZIP tidak cocok; pipeline belum mengubah atau mengekstrak dataset. "
        f"expected={EXPECTED_SHA256}, actual={actual_sha256}"
    )
print("Dataset package SHA-256 valid:", actual_sha256)

import torch

if not torch.cuda.is_available():
    raise SystemExit(
        "CUDA tidak tersedia. Di VS Code: jalankan Colab: Remove Server, "
        "tutup dan buka kembali notebook, lalu pilih Select Kernel > Colab > "
        "New Colab Server > GPU > T4. Jangan pilih Auto Connect atau kernel lokal."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print("GPU:", gpu_name)
print("VRAM GB:", round(vram_gb, 2))
subprocess.run(["nvidia-smi"], check=False)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "nbformat>=5.10", "pyyaml", "safetensors", "Pillow"],
    check=True,
)

import nbformat
import yaml
from PIL import Image
from safetensors import safe_open


In [ ]:
import base64
import gzip

DRIVE_ROOT = Path("/content/drive/MyDrive/skripsi_lora")
OUTPUTS_DIR = DRIVE_ROOT / "outputs"
EVALUATION_DIR = DRIVE_ROOT / "evaluation/multicrop_v1"
LOGS_DIR = DRIVE_ROOT / "logs"
RUNTIME_NOTEBOOKS_DIR = Path("/content/taniadaptif_pipeline_notebooks")
PIPELINE_STATE_PATH = LOGS_DIR / "complete_pipeline_state.json"

for directory in (
    OUTPUTS_DIR / "smoke_test",
    OUTPUTS_DIR / "multicrop_v1",
    EVALUATION_DIR,
    LOGS_DIR,
    RUNTIME_NOTEBOOKS_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

# Payload diisi dari tiga notebook tahap repository dan diverifikasi SHA-256.
EMBEDDED_NOTEBOOKS_GZIP_BASE64 = {'01_prepare_public_multicrop_dataset_colab.ipynb': 'H4sIAAAAAAAC/80caW8bufV7gf4HrvphNYV2bMeO1/HWBdxYm3qbxIatXRSNjSk1Q0mM5sIcTrSC/nvf4zEHhyNLmwRoPjgSh3x8F9/J0frPfyIDn4VhPjgnH+ALWeMfOeYVq5TB+CCi2TJIPsWDkXzIAxw9PTnzDw9/PNGjEStoQAsKz9YbNZYnZeYzDRsG/kImNOaXAU0LPiM/kLcJDcjPPKYheVeGBfezJCVXACVnBbml/pLO2YPemAwaH98nBZsmyZLwmJMFjVeURCyaUp+SlC5h9UxA/e+Bn8QFi4uDQEL1Ir2P93Tk/s7T/zaArmg8JwUL6YKESZjkhJYBL0YkY0+cfSIRjcuc0xHJ0xCHNcnkiYYcPvAkHpGAxg2ItCySiBYsALB54Va0LVn2kcJYQVMScJKVccEjRl4nIZ3+RN4kyTxk5CrjT0wS1wAZsXjFo5TGJCmLtCxIyrKc50DkiIQ0TTIqsSDLJJ7xeZnRXGzgAgwB4hH/ShHZxO0nAdNCZZ+ZXyJdnp8AivA4LsOwpQdn07PDWXDEtumBRFQo2WOfagBJSVYAtfki5NMGverBxzyJu6NJ3h3LF4ByaBlfWSaDBsx42FSyWZZExE/CkPlIeU7UzNfIAZaZM1NaIMJ61i18tWssKPzD4PXN28t/eHfjt+PL+7E3uXzzMCBxUoAWAykui594lsTnjVX4J6M8Z+R+BTKOxp95MXwY3GYspZlQOdCfaZgseXaOkMxDATiXS61QQr3ch4Fjx/D28vW/Lt+MvdvLyT/JhSAFtnr2BLXh/Xr79ubyCmDcTe69q+u7HQF5ZRqCMfCAqiJvQxz/+3b8ejK+8u7/efni5SkAfBjMTik9PD08fXnmz169ZCdnr16dzo4OZ4ezWcBeHB6+Oj4Ljk8Og+DoJTudnh6CSvrs5XT60mf+bHp8/DCws6DxMWAzUCUKO3qoIUMUtGOKJuBzONeAktJbV64YOsa8T7xYSD4IMG6SshhYkk2BVEJzFFYQMhM6/pslGfEXZYxCJRwUcBjSaBrQc7XGzRgNhkeHL07IXwn+54wIQAW4Nmg1zm6ZghzYUMA20c1YUWaxnrlgn+WnobMT12iegykOmafMGghVWPKh+t9DHoyIELUX8GxE2OcUThsLPMm+DurNhVqhmmNOZ74CXU9WA+ZMOJVNQC7PpbRBKnFgyL+xH7m4MJG2sVvxsU33A5hVsNc8nnvi9DyDkZicK4TQVtjxtW5vGo7Zw+B+cnN7LqwWUeeQTMsluAsEAwjPy5CCKVk3d9n0mgz8A26Rz0A7tHRq5h8Arepc60musOMWmpGyFqQ9SetRdk3wf65vVUgwZWEZgevNchZwKjzlLAkDlpE0KZJ4Dt8lzqQAp74kHWSrk/kwgOiARYJ74MTXFeEbl/wqQSjIvTAAxHPGEMKMkIYlJ7/dgwEPGBl/hqcZQF2ybYC1ucWoICzJRxrSGDEF+UKUo12Fa4VgnpIiW9mEoOUFQkexuoh0PmxLEc2TV7DPxZDFEFqA1l+AUhSzH85Aqcx92GefpQUZCmi/3N+8v2IYjoyzLAErcXMvPgiDCTP30/gKV7uURQgHag9wUduJ8O3wpV/tpXAuKsDunKFntjgwSZk2Fvx31l0lwyHx0JuuIFbsQoAzMtwiAw3IX7CIek8YE+JBc8h3F+TIsg7cih0HPHMxjZha27I2YnwPWNIyKkiGvbTDEcFQzuO8oLHPhi22jcAFFo59XZu/f7sghzvBF9KCEwIm1ulfIGW6C904U8bKimjgpdzEBL+jVVPaC7F8kYGa7qLF7lZrrXIVpq3LBcTkxpRkmrPsCWaguHEG2KdORINxCVo/DEsEJBs9CADW4zzFIK1ZoM0NMeAESFq4XziEhcCC90lsU7T6fLShts7NH4QtRCdV0wBeKfEfA9xzcAV/2tqIzIHsssicnvnAcxHO4ERHnEbUsZ5TqRUYZ2F+0ZLqlvltnGrW9h4/tbAhnJ7z138GFeN7ie8NZntPDHi+bPVHzktLHywhTQ+3VQhTrWpEkkByPQxEw2GC/6SlAvHVfNuZyDqOM8hacPTuB5I6sD3zJU3PZWRii+T0BtQvShp67SPQDn4VAMdOugXAdxfNI7W7/OzzVMAlqQYf5y9ziOLsUvUTP1kC1SipzU/9MZKCqV3HxbqB72akaLpYd2nb9IK0OpHWwXNpEMjza5lqGGeXppAnBnbumyEawwIEzVa1zjYcN+afYn9UHXfdceobVyVtEJ9ZQ482+G5SYjz/krREYbICtxPBKaYZbWQoINb2VhaN7olWDRTLGIhdDiOe55iKJcuLSVZapSKSd2O1SuA/6QQe0ARv0afls6YJQGNsCPp8i4aK3Wv7YVYOZKS1DYBwnKIm5vpJukIuJtOPQ7lwpDAfCWtRLC6alYR+S6GiWIMpbdu2ZbXNxLRhbbcz2nS2oz5gcnuD73ZK0vc1Q1pZFzTnIQFxrMq8jEE/Mcds+pVnLQ9iXdkX/LI5WLdI2oyehyHoqqFIA9WE80dMVg55WxpS3xTLiGwru4gEO/w6h89eIeip1WD00Vuq2U0LegMJeVRzLsQLkbBoD9AiibhPFJPInM5paIkj0gxCJjRvuHbKMqkyAc9RY0hAM07WmCEY5sDZVJ7t/GGwnec9JSZdg6sqqs9W7prV3xHR367f3/468d7dXI3huPYW9gyc2qDMivCIGBXdxuomdZJ5DwPVBwKzid2OKAmYYEoXQ+viVbFIJBfzVe6qvNj5Nr0Q+iqYHgezV1/aCxHFh7mo2YPJDmnVXQiwI9RTeMVHboSotWrtYsXgGxE8e3V0cvQi+PFLCb66u/5t7N3d3Exs7QIk4eDdSrTDDvJlxtOceyFYE7NPMLm7fD3pAwPJBqey8+jpopvS3zYcrVgKTgvsQU/FrnW+7n59P7l+N/auLieX9+NJTxfEgs9BG2SPOcRgIgMrVGB3EGIJ8/Q9aJYf5FGyZB52HkH/e2e1N+3OY2CdStFqsj0Nk3neHu/EfMOGeA8q1B03WkJGhcEtsCMXDgA7ARBYPuMPlKursoB2HNE0Pk67/2Yu/O7CNEXPt966nqOTldRxgMhE6q6vHAw4W+ZYzekJEFopiYFfNy3Ru3YdvM0cImoA6QdB6mBkcuQbmYnjGT08OT2cfXGPeNY6jVUWYkpNRbtZVGSMDZtL+uyFRRd7lE+E46pr7P6Hpz93dA6jcpr5C9BxEzE/ybIyBdMBbpRhc0rNc/GMAtChJfNqr9k1n0JBZ2UO6pYCq4laTNZtaBtboRpU0y+8LEngD8uTEFJXwwy6atxaFFS0gWHSxPF4lmCF1Z4NyvwDdhgallYCcnU92undVaeoEpAI92w0VKmqmIfRozZFXpEMrWucfSoyaOAbZ59GEL2RtUFFTwEGjVO86ss6MIRzfTAh2E6CcBd4u4YUkH7CIwz5/BOLn+RH8GYYNMgm2+aZbBThCK/U4XRfwXsvflxDzFywLIQEPp6TRquMhlQYyZ24o5VIyQfyi94D3bwkY/QkWz4dnbgeeHf5/vrn8f2k25fcrydpu6Nh26LyAYobRtLwVTtqQIHRnmi3hfpCmR1o+w0BqZtWkqKc5SXlBjnS7QhKgjJKa0qwlhyAqb14AS4/zsuMeTT3Ob/4mYY5c76RH5pOGZv+ePLFd5Uqpzy5u37zZnx3D+JadyIjMLwcmAmfmqGeHO3GUR/pvMTamzlfj3dX+HRKLRuo4e78JfXhIHowly66y9pPm6s3W1WqyPh8DtqgdKrDmh2UaSJB6By2pVCtW2yivmCoGGCE7SmzD1dOQ+57SRyuPD/EvBWDVBCpI7Bc1/we1ZwcdZi02QH7f7Ep/0jRsC3BneREbv0Dbr31aIAeeFKzrF3k+ikitt4YNDcntGjG2YcO+Rs5kn2HzrQw8SHio3wZsbCevAOhOoqV1ygQMllQiDHQUUN+oAgXFyqae/S3I9MsAbRyFuibOqaRria0DDO4APDSn5Br66ZuzrOkTDuj6OzEfU2ZKoG+Uh5LWdfXNi23pbRtqMrpLWQPFEzEUs8U9j3svdvShLd/ffydvmtq+A/hQxuQLR5U8eqDZaeGjwl5bC1+I//wGbKvTcNWP+QK/uDCvC9ew4cuJEKWsNfsSGuRfxBQH4Ea/GYyOsICGLI1t9hjGUcJtOugzypUx6W5lyY5/zzsY4gu4/cByOZhMgVl+6v7MbX0UzZ9ulYhD+R9eBjgV092zB/FvjCM2yLxm66aNRkARq4NdVdF+6WMQro4gF0pmdNoSjNxoqO2/onEVqYWa0HzxlL9NDC2YSBss8cDKVFlofSgPKX2gFldMMFAoLW2Mb5tucgkDU8lIOi9HX0ttwaNql7B3CcQ1v7NR09bhYDSK5wLWYsdvq/E/b3TkyZ8yhJ0Tsrl9hxq2QITU7YkAEmxYJn0jSM9HentsgVNjf0QK042QKHeVTLFhKsBuYeTFriPdom1qf9KHWUtnFx0zpVKW0UCovp+RL53PyY8HrZxcfZrrwAtNTPwrQJwlvIdAw9baGVeBegrdP/76NobeWTlJUf0EhlYBSL32E/XsPSgJSkTzt1CJnlxtnE8BSE6ONnrioMAQtIEVq6Me3oatc3BurFXPykte9FESKD7MNDx0BZFbWTge9xGea0OvYqOJBXTJGQLHTctQ8ojcmkLljRcGdq4EH0FbEYhWRtKBxGUYHt9WjBPzAA3MZKXtRxxzUB6o97XEbDigN0nMS13yN8hZqzCpRw5ofbFAjD68V2y3iuNklwMpPrYl5V9fBF8CX9Sx1798aGUscjm0NapFzAU6bWTqFyjQFmHChXSphtyukxobvRBA36EsPjsUN6/b5yDjn18nimVV1Wh87KUFRFs/8EOqbjKC1uAWuNtshY65ums5DPqUqwM9XlfE/K1sNABx5hrKlJlMlTe2xHVYLH7s2yuGOnYKszybS6/s1W9QYu8npSoK/VuUPvB8PZ76oGywgaN0jaIHP3ZtiQkQBKBHkK79Jjlmb5pInx7NiPc5TQqxCB5zktRlxV5WzOIC5i4QWWtVrnfrHN48ur0lE5Pz75S53B8e1O/b9RqN8kOVfXOUVq/OPVgNtVUARnjeksLb0Q6e3UkUFWhe1sT3faEXNBjAhv3dsTkVr4xIhY0HfvqF+0ionUp1txblUbzmpqJTSdhB+Ym4raBjVn7APJF1ccOqD1hV3ptYLZTqy85NGqy3SKfjtfOdT3BVnnTyqdvEp+bfbeRpXgo73NUFdv6DYYPjTruY3elRltvCorC/CLJVggDrIFNt50uGOGqhQXFdRgmVEl4VUR5tKyrHXvf4mbZxQah6R9weddp9FUnh3al7vLEeBvJcT9l4DxlMcPAp1G8NhWiLmI7WMU2ayA9/mNoV+j/JxQrFyeMJQuq95MVcsLFPWN7NIw7aRLOe23CN3AuPx5Pg9Ojmf9V7uFcyytELvj4NKSr6iqO/Gq+Dnx7/VbPuI56315/NqjsdLRplIZY6YhR/r16rs4lfK7qGK5RihqJ1xQsfWa5hf0tRlUxU69zyJmtQtozCD1TUpMA0T5oCvFjI16zmJ46fGtVySB+a6FrvdkvVFNzaKTp2T69yuTkXbIK4w86S3y0rVVaMhS6IO/uyqWim81/Z8OheO0BX3rduetl/0GGgB2Bzgcv9vpBhtvGbyXkXPwYAeT4opshUtTqjXLBdq4LJK3fKyD4swlNL0wj6pJf+LKeF/AFTct8pO/pmy9cYkZYvaDYDNEAqgS6xB93CPlP8pZNmoBIINPNCiouBOKdyvavJsifVsAevHoFHthdRgx/BKHmMPwRZ7/FK3wC22UxC/OU+XoIGCxlKdQKGS7tAjnWDMf6UQlixmepeKaf6BVy9HhQi7da5OEliXqv9goNR7l9caXGPToaaCIErEE8hRMTUbSIJ60BL+JxksHwyz//Cab/DwIhQID2QwAA',
 '02_train_public_multicrop_lora_sd15_colab.ipynb': 'H4sIAAAAAAAC/+19bXPbOJLw96u6/8D11txIMzJjOc7LaEez5U2cjHeS2Gc7u3dnu1iQBEmMKFLLFzseP/7vT3cDJAEQoKQ4uX2eqs2HRCHBRqPRaHQ3uhv3//5v3s6YR1G2M/Au4T/ePf4lngX53YrD850lSxeT5Dbe6YmX4QSfjl4cvNzj/Z/Kp0ueswnLGby7f5DPsqRIx7yEDQ/+6F2wODycsFUeTr1d7yJlYey9L6I8FG3F73GarLx3ydmhd/7a6/vPrsq+vR3lJ30cxjMvY3khms/YqIhnLPaKOC8W3opNwp73ic3gYc8bsxGD/07g9YKNGXyYs5jNfQXkX1nE4gU0mLP4juH3zHubJLOIe6+SiI28t6cffe+3JJ6GsyJlWej1n+/tefDRbMHmHjSP2FyBN2IZj8KYe3yRrXgaLnnc80YF9rCIWLj0FgqonKeA4ALxoc+v8W9BStu0jJMJL4nPP/NxkYdJHIwTGDq8joso0uZrf3owPXg2GrXNV1LkqyInZrh2TWG4XCVp7o2zG2Wc8uGcZfMoHDVffMqSuPl0yfJ582mSNZ+lvPksm8OII8vzYrRKkzHPLHCyO8vDnC9X0zCydJHDhClPp2myBJ7IcYyebHKqj0GFMfWudl6dvDv8S3B29O7o8PwouDh8e7XjxUnuAdcnmc/jmzBN4oHyFf4FfJ1x7/wuA8yOPod556rm9Uk4ipJFmA4QDB8lyQJghZJfBderDOtf7XTt6K3SMEbIp3f5HFHY6SF1/BueZsBH6kc1Qf20iDuXVzuA9iRku9kyvNq5hoU15+PF8A2LMt79Nsw73h+N90Yv+Vdi3jxJx+5Zw/mhFv64mDA/zAJ2w0IgZsQ73fVzpb/HP8AGH18fAjdN2AJXecaBer73OvT+dg7TNOEDkFFS8NCsDbwzvkxuAC5PYUJ6AMEKNi/yYkUCDWWKt+DLEYvCijN6IJiiwluFUTgHUBEf595vPI155P0i5dkvLtAf+K1sInCAliD74O+LA9/7KyMhKwAfFnkCTeMYwcOcFIAH9QF8yiK/Ad/Bj7NVEcRsyb2hSvwZz4MJvwnHnF529tSvb1K2DGYj5xfAtSB085Bn8J2fJzmLgiUQNr3znnid/t7+wQ8/PO1a1gSMlBZEiZOtzd/ODt97b/9C7VLg20lHotPz9rtdnaNKRH/2+gcmA5Xw/n549uH4w9uBtiWwWxbBks+LT7RniPV9cQC7ztu/CFpnsBJSJpf5N1h5k+fTvYODveePXXkkO2ckmfwx8ZVci5M0vOF2lqBX/hJRAwI9GSdxzuP8ifjiWw24338x7R+MHj3g12fHfzsKzk5OLoA/cZdoDOHJ+7vX9G+2SMNVFgZRkjJdXr86+fDm+G3w+vgMgCgQn8AKjZJZ9gQxAxYAVucr4AMcGnYCDJRpC08COj28+BUgKWARUi43l2BVjKJwHCxLHcy/Y8tIg3P28cPF8fuj4PXhBWxoFxIzc3igV4VMKHmBxPBJBTS46euDNGG+P/xw/OboHOlm6w4xhn0I9+YSeLCEDqc8y31SNFoQPg9eHb769WgDvOcV6lkwZrDF6Ugf/dfp0auLo9cVaue/Hu4/ew5Ar3amzxnbe773/NnL8fSnZ/zg5U8/PZ/296Z70+mE7+/t/fT05eTpwd5k0n/Gn4+e70GXYw6q2bMxH09HT59qA6g6evXu8BzQP4EBnUM394YcudpBbfcKOPCnZ73GO6ED49v+3l7zNanG9O2L5kuhLAekLDfbPDQ3T4XZcPdE3epLN05V1S7Z1BvxqFhWGynIR4/Wke/ayyqtvlKZlOUC1sNyxUIQpBHPGGnhER/BvjZh8yIq2vcvOWAXBz929OcXJ6cDT7IhatdhpKEuNArYH/iS9geghFwZUvtzUeSUZXmIX9QSqVxJyjL1fw9XoE4DIQDybrGKEjaROoUD7qdWSpM5VSJ4x8hyW7J2CrtWOqwAXOs+4pR1nBOQcjYJcv457/AYNgLgnuHVTpFPd1/CcjZm0pwDp5ABFQOkRpaD2QlStuv9ARf9DWhek8ZQknQdmPJxNmcgPyQ4h3TZGjgtmSBNbks8n/b3twZCI6MZVCG9eLE1oHHEskzsvJk5UFW6KYAftXAqXilXUMl7Yt1kPCtgwa/YAl5NwxiU1bUSpEBT387dE5aG3v8cnwrOJpsoK5a7aJcRBTfWhFUNiE89SX4elNTNVhGMm/6GCUnynsc/r0D55hOanwbJ6pblllc/6RptS20nQDMXmivf4r5bvqZ9NmoMSIpDDYhbBlqnc1pNngRiTFol7EAd0/p5cFq6+BdP0yTNYECX1yZDAclszzPOY0KcLABsAhzUcdILvTwExzJI8gBNYX3QDzD9dQK1yiifZoCgd7pIYPzpZzlojA1kzCGUXQZxsRyhGVn2z+EBB7blHR3/ngdCLc2Hfetc5emd7bEkoi6QEVzX0ph/HvOV8Af5fz0/+fCao3p+hLPjsQxfu7oQM+iz1YqDrQVsQoO5V8b3MPAQpGQUWjZe5x5APnR13lDB4v4XxgW3vJbcHGZhDGSJx7wDw+wBB47z7qOwrFh7ztIiA3UmLVbMS0af0ILGIXwRusjIZcfwu2UxiIV3W3M3Mnee4ldSUldv0LqFZQ57pYPn8E+cpEug9u8ggFSQWhfA5asI1FwAfoV/CO4T+0BTHoFIveGlECKZZeuk6541W/NHTVo9skUC3Dv7UpbquJaQOmhyOo2yJCpgjbr6gSVztePDtoJLWv8cNqU8c3+mN2ZZsEqy8HOHdmQb4SyQul+JmGKxghqIAl3jmIcvX7QWdgQSGRL9Kw1gAnoxqNI5oG/r2DEKAxefTSZrOdwCRmhUIOfMBVy+2Gj9yiVjYaEms9Djy71rZJWyk80o6WhEG5WdyKBdTcPPohvJKONknCxcpAYV5b7E6cGqyZlaV0WBJZtxq86jjb5lk6gAtOs7GzPYjC1HLLUoPXVPDtbiEeCk4IN2SgenP8iAXt7PQ2/va2AmpGAbQkYnLIqSW5gvUHR4jGcMGXkxdvxPq5lgU/jFy5+ruPx1y0erK83NQNDGecGigDrPLO4Qcq0K7UpOYJ6oeq8i9SxfouZE3IBio/rIT2dRMoLV9YOd8kB3nQM8Fk/Eo6yYAiv7YzBDpkk0QVUutlDEAPrg0jbrYW+8bzs3YBosKnC4i4AO0eg0zDJ0DlZ9ZkkKtkbHRGZXnxOTQEUchRnaKCYcfSZ3zUF2mzZGxOMOWToohDTbx8bYDab+VCwjNq+1sPsK3oMK0LvXQFtWG6CiE2d997UJBf3HRfypWJQLisxGuR1O2ABmx/vRKxdGEsYdva/LQX/vumtBySD0Jjhpwibn6ZjFebFEd1KJrQUdox8nPqIzGxoTAB5G5DK9uop3K8jii8vBPgDcymY07fx7Ze2KHUIMcQFLDXGPB9TvvUCkXXylPC9SsUJ6VmlztUO9oYfU6LZnba0ZgfQVLGXtYXfNh6XLaFAevPviScdiX47ucrQgu/6cf56EMF+wJ6wBL5w9g3q52dsL+Y+Sp2qtCwL7Z6X7bFD7znot4s/+U3OJBNKUv39wt+B45tRohBJQNKDp0mUKysVO6UvDBfC0v9/teZpjDB+/eNF1ul6QZ1QMAAGHW8dCK/shiIKv1c5WRtBr8Tupjs4a28sa+rWiEd1m7V8K6to/pneOs8/Xct2iwheFs3lOp6rkJJgUy1XWUWH0aOMIFvwuG16kBe86Fm0yyvDkehKobkf3zJdT3SSGD1NV4OIZWPwrcuOsWw++mZpuHdBlCQJJbW1BXZWtet5eF+R4X/d820nlcs9+iUu2FtEhDnBUZKG3AM0sc/n5ms5ZUDWteD5Y/arf4GT4J/7T3t7+U/bok+HjN28+nh+dnTtOISfhdFrAQDN9P6o/Ozt6Q5vmzZ7/9Jnfv7LEr2h9+PwzUN3CwEZkj2XeLh2WAugMuNX1nK/HURLztga7u6OUxeO5u4024DZAE77K52199dtezvN8lQ2ePIEBzYuRP06WT+bFbAaa1hTU5Xo2/NYR47rWiG7d9q5tD0X4FIqyNkHNo4z/L87f7qvHjlXAmfJ8PG/ng8dNX5KGM9qXnS1yNnsEkz1iyjadIjkXJdVt9MUVhX2CfJEtd1FpJdLqY3gEvsrPP4K++48iTDlYKbCF5J9zcXrLP7PlKuLe63JVeFIIgRxf7q5gJ1zxaT4c7vkvfC3O4I8g0ScUJhvXkp7N6mNgtIeyOzxMwyYI5ReA0gcwvneMDvgI3X4ZaWGab/OPAsYimfOUo5dvDEAo4BbskzD32AxMm1W4khsLml2AAHZyevTmAk91OJqD8G+x8rWQGQzdvDg7PMa4LCDtf348Pjt6fyTCLzqNMAk2hs2FTlUQ8ad9GP/P+03GI0UyzkB3WAL5fhke+Ad9f7/38zNbU5UMvZ/7tiaKrNoF+UV9HzhblyEtvwz3/f5Pft/Rb8Zg3cJ8JIghAMSGVnijMM9YPCHrAlsePPWfOppO8+ndL8PnvZ9fWIlCvY0Slk4IN2j31Nbur2H8ie3/MiSUDvQWKi9//AArAvSWkw9v3h2/usAZPD189dvh2yMxezsUMsgSsZpucQz6dtu+cG1yFcNnhRqBoaJ2q2d3aRdFMNXhyvWKzp+iyPV6d7dYzVI2cWy3P7Qwstm+ITzsgqOrLb8LpOThCcWh/v0//uKN2C0rI1krByYocqDOJRGf13E88Lfv/cYnBQUv5yDPVhrgSQgyDhYyBueMYBmH8EXOZxQMlKzQYYbRkdAVBS3rkSWlob8EEzgc8UgHzOApSoLZDMBGKAvKAHEugovKiP4KVwwkpWCV/yc5pIjX8sgdmudW/mhZKl+BP9ZQySBJSYBquBr741tu3yG/ALNKswOVf7kM0ShXkKXvA6HPN+d2sy075Te7K5Zm0h3769HhawyYN6Dh0X8TUYshWJnL1eYrECdr2RxN1xVRgj4/Ho/vMDx6BHauyLsZeikaoLotIdaE+AfdS/aECyO5gyJ9628q+062lgkGzbSK8YLhRuZnKz4OpyEOT35yXj4557nDD6QoK9bYyHpzFh6nLbbo8oO1G3UNefPtWsFm/aZdNt5061aAb7SB1+2t27jmVuNZEoEljsSmo/VyXjt0akmukfLcVZ0cLVo0FjEazfkSEO/BwpXMAoiVPV6SX0msLoI7ER5XySIPFh+NcOpVTUycfDxdyzoW17XeZ5m4o7Jjp4LadcfEyoHa/SZnYteiEBj0ZB/HY7lpVUln1ZoNeYYHbgKc4a7GgRYxSIBJINYS0d6l6Ayag9WXuT8N40mAo+voULtemBElPoCB7w7p0oblsL1gtPc68AdvWWS5N+IeG2UwN/BrzKCBl8+h12ozniRc4ICvUBDbAhQc+T0y66DysTRe0VJupoIZKUMktCoolaSiXUzIZ+Db03BF6X8NYQddlJ+8S1L2iqKcezBhGAIUUAg6T4MwDpbJRNNdlJ9lKxLjVQ5MHPvnwNtAu5BFnerZO8CCpZ2Dnnegh73aezRnrEaykw73ex5mKgQsWs0Z/i9nKWbdwJdFxLMhrNu9q52HhlNAQ9i1KUv/FWiEHUxxgG5EfAsJk3oxB3R+qgKkkxZkpBT+xdjwTre7wYq72iErUIKSM0DBykRWUEEJn3HKYe8QSaZ1F84Do0pwXO0o3jx00dbupCCQ0i0I7F8ilwSCS8S3+GDDzzTSiK8xQSDLuB6jrHxXyQnRWAg70rzc30hxsu4Lobgo/vwSRLsv39RHTN1FV9g21C7HpCzZtaDNVDPSLoG8RcqlftiqZgJHm9354jAR/c/oZd/7siDnv9EJEhg+NXi5YZBVU9kypdj0r65iq6j8sYlglk/Q1bRpW56mbTH8crYD5VPJxJkeNurAo4phUINhL3f71zbt+KxUTertQt0/jZOlNtxQIMOTfKgn9JEIV9SsynaER0ECUOwyocqyysYwGLQ29IMAjK+W7rXsCTJekCfiCPWJCOPXnlGymL+6szn/jZ62yEKBHfkoztO7VQK0rAlYGxzNeCSjM0Mrqd7yCuyXrl00sYzOusITOufRalO7avvF6xzC11nFx1LrIKJwC/H1xZwtkwWoQ5hU4V7ObpRb1nXrR+0LvFx6VZa6Mo5QGd/AK7egb3REh4til6V5CPZVvosHain8+IoHdkdvDj++uwjenZwdBm+O3x19OHx/JPbWO9o/aVkGtxwPsTNfs8UUer0//lCDCP7y3xfklMSEZFuji6MP5ydn4vAVmr1US2DUTr3Tk7OL4OKEkFGcqVq/VfN3J2/f4r/iANL9wem7w1dHv568e310Frw/PPsNaCC8pxRKNk8iYAxhWIA0Xd55JenFs+wuzuc8D8fK881yXkCNwPgFQU2Fih0Ki2mkUCth4tRg4+QPAVbwE7zfM+1QMHGWq/wuWNOu8nEY0QXk8wt/5yL6xviwaYFRtE55VlvHDoL4noSpPZ7UjOQSQYuZh1mRKVAwSe+acWsUIYobxfp4VRM+tqzNL8LVBj+zWoYaKSyxqfaQSuWrzWNXJaZACZrAlkhZFbxnWZgbx8qWHeZJAlIalGAyV5gHlkMkDIfSyOjc150+ePRPtzX2TE6XO5KuJTXnNgSOqFQTWh89b4oWzG2SLobAMrnQiLFkAvx3vCowKJRlWFNkEjmTJvAP6u0wlRgH2BGtfXzU6XZbPlIWi/ymfAJMDxRrrCD1j7EQMdgMe2zrD2cB2qDBiE3bxiNpLdFCm1b0h310fUyaigDHX9xMqP6xC48fh1rwjaV3nOlNEDUZ/hzm+EJqpLjzwVaMq9SrNqA2IkmRYBV3w/Xj3Q4XitP8YnyUVarujFuiuH7+9FHMmVjZU35b4VWSllgMdjpvLLPVwfigBe/MQTC8YLVgsA4Z89fWANpkwZEBUdpbwvopX5l2OMqDKSuifAhad1cJW29nXfTaYLUw4WPV+0Ua1e8sikX3MUxWnzKgwYYOWdwDFS3FoYIYeYlH9A+6f7ZMRtQYnsW4hkGdR4nLJwOPkhA3iS62xxXL8NiBsgk4TgiVSOKWAOJ696HG1f/srVVOxPbq/+1fWJlY4G950R6FjJ9VXGptKQiC7eykedhI4xSR00ITsiqZImiaNi0t1trkJtpva1XUp133aicdrd1VcYWM50WMFb08sLDTTsSWI0wFkBsSBnJT2SLvBzIWwPwdUdSoi00Fzn6xwkDjDsHu2hlPtlSCwzeiWpWGFdTHj6izd4RZFVRKKFbfg9kPxywiGgsvrjyFkd434QE0x2JCKhV987nl+KZjzVXVMnptWKHIs4XfpsIT3PjCBxmQ4bTDNOsmnwMM4d+E0yUXNiXWNd6ZxvemBy87r0pQYkcq4dXnLJ4qurBeIQ2pVVjV6EnLqzFHT9YPoW5QhcQ7rT69Q8tMm8AuK6F5baWUYHodbK8BpUceOVBu0StSvbzacVNG7potdFnjS3DAW08iteeu3XzQga0hEZ1+qqvTmfq/8XGfLCyXyNR8YRLJXZls1rLKnt+uOE2xXlE5U0LmD+8bHPC9ePP99cOf1oKrhVdJoxKsQbMa6FbppVvwm0zyamMyjd+bPgNdSG1agsM8hjbkRulXoLM5GM+siFhK0gS0G723dkVnwoXrNkl7VHgT+Di9qwr/yUqc/nKBTtOVjZNgMQ0bu4st+ZOSh4cwFP++KYwefHswlMjTBDPYHyerO5qIdRG+SeaPoyTjnXpo3YabqRyo6qrSh9/4xm7Si6qnhN2+tuh7Ri+OBFVVzzE+wJ1Hfa2JlG2W/85FCbhmRA8xJt2pdh8tWT6eK4cLYlp9+75ZI7teGG5ECTopMYCuEYru5eIccZhV4R/efaM7VZ5YlYWsStvVx9Tz2vdEypiJ7hySXYeliw/j3dpsdqM9ZoXGi85G2sNjN/6SQdkUj7qJ3I/bIbdVIzaRnHWSoDeF/Y3MwZbNypplrO7e9WC3x04NHZTwBJK3LJMlINBVwyce9WLwsm/FTZUYxnRtLVA20WBtIqRCewNZssVerBSZaGgItEF/gxOsly/7fDrtv/hKlY6pgKd1KQpfFSwYbEJmS4Bn8B21eOPG1fPEAXUFUvy4rCuLAosqrc/fn/x2FFyIEp9o9Kn1Oo/OP74/Ct6cnbwPXv169Oq305NjOvbCcDNnEjBFLQXJDSyjcIKaRF1cW2ZaXhx+OD58fXh6cfwmOPv4IXh/8vqoyrqs4gusji5UhRtdyFjAe/yezrrwXFb8XFKe8MMmh+02rNT6VVgTx+z6D6khJGz4mZ1rJLcQbFgPQQ/cKJZb0NU6dw0qG6jrfZiIO/nB+M6VxvSRSiZXh+kFBvH3vGLE5ipNljz+xCahRzXMZU6ChyTykjwBGRPqCUdLvqR0BooRF8fcwqahEv5p4bEVG4Etg+YNiIuF2I6NuqW+XqVWzL0yDyhcFRTxUK1iLp2EDhqB6mGL0XRS9GonYhhYcOVOtq+XOFZB66jL3pjWekTKkBr5i/jYIjbEc4GLraidPM7BahrscyCwgGW1ygSv1dNCoSK2F1kgyoFH4RLjR+27UTXeS+iQUtkVfOmZ01dVVVpUv7jaEQWwAyr/HCgntI06FxISKTGVz8ZAQMZHVv4VjVS2M9gmehUk8QpAfAUkrUBbUBXzRycFAjVX1WdZgUH51CxXaYBaU7ISJzhKZrJXS3ltWIRPkMtUfnQsS2d165roYNXiOf6KpRh1Ls9ZSPcOEpGp0rUjt92H7UVtDf+nbQqtHlDp/bS139z/Wfo+bVBq76ft7XYlYXc+YEIpHciTzljrsZTEKgWOss+6/Z0kuQkXjAYMMapaYFbr9WU4tHRlBUwEEA/d3nHnuuxtsMJ6DYVWc6LTFuYKZzv/7fi0ygQkY8k6IsWy0LlJ+951k4HNAPpTvQXXYX9VNOEnrbwuhWjWdO+2XtRRL0X75lLWAC4lhLHKpSr9pF7hT6o4OHdRecUss/SyZcVbJWyOtBPlipeqgHEd9ddQIhqLTMdIj6+1Ybuxkl8Dx2mx5EDjH31b0Tby1q0FjwxcX7XsHdaU+wpHhaXbPAP1B3ZQVcWyy/pcTy8Rp5G1WTZbAEC2vrzuXtsZyUZMK9y67rhQrYox3XVk80ejP8cKQ2ZsSNWIIPX39hwQhCKTpx0rKKKmOHgua9gQvJqorYhtJHnUzzDkR50W6uzA0ZhFUccoydecN3uduUcv3lGxyMPmygX8Zwu2Ere3SNege/uRtp1YGpLisJEEL0chJXksLbqrw3MsPQJqwp/n/RF47x9s4L052DPDkmRIREcEKJM3puednNOPbkt8hNgGpsqNNi93oVPvELCtJP6kWBTx7E+VJYUv/w64sYyVNZsdfmBFK2+QAnV02v/UNClRi6wyMpq5oaU5jCEEpdliSZRk0JdIx4J2FQbq42trIqmikJaxGfrTrjPjM6gKYlEGpK3YliXDtHlxycARWKLGi7gvSnFEkYABDhiOOCwoHtSOnoFnC7mXQR1Ip2BejARiQRGP57DhixRO22cPzeHVylKJef3EQklj7xhspFLZTgQbgNac5NpQodRf93SIKQXLlMjRCO2201RCLQliC/HegKpqioyabWtNi2mhlsg9RwjmM8fakASpV7Ur41iZZtLc5LqutDUpIoWu1vVv0zDnQq0xOlbC3zTZoGT7gHXVUIXa8xX1iouuelsbVkqrtpQqqk1X+hpXH3zhzTdbXXVjs7plwUVL/cDKWL+2fVddlqAmYmFtfVDIMDGHZeMwlBcKqlX2tI6vm/n/oMem4WzGU3sRAPV6J3FZUq+1TXlpUnsreXdSeyP9DiUXj5uRmlQXtSr/qlFPnZqyMKwyeFlbVz4RAaryd539rpCL8pzEe+nZ1nAxON7scAOL/L3OyktYYGxRrMpeW8rdlp246jy6+MNESinC2FKA0TQAyKimwxztO+W561M8G1O/JgdrgidMLFwsUV8gFbVC6w/D+g6vTaveitodqyQKx3fGuUFVCP3JvYKG/XyxwkGcaFR8X98M1mvcA/YglHhlhGIAdIXc5kP4jYpAiq92kzi6Q5UwjMIFqM5UcfwRw2qQv0SOUIctpJwHUfm6tI2A3+szi/pEaEN74BUTAcSiL4n9iAofwcgKlsubf+l2TO+QULCfrNYogclV0MnkTchvA+PiKSrssym534pKz2IRYhpnCtaPJwB7ohs8c5I9f1/VE/++a5C4kfInlhzQDQ00Q1Z9o2y/6TN2MD14/K2wdXGY2kwwi3IqbfIDeS9iVzMzDBgtGohIbMcyHwW0kOdpWIpk4L07eXX4LngPqvfxhyOtVpEom4olfZFaA+/7Dyffqw2SW5CoGSjk0/5zeBsn3xuXq4aTbIAGsbpxgBaOtf1TUBIGWnLakhKOS91qCoo6TgV6pMJY23k+Y82FlI/DjN5PV331lrK4WAayE+i8b7yRbkDzVTr5/SYYsfECFE/U8EFZGasFpYAdg5jnmL40gP1DOz6GyUaCDvRUP3yKpuI4KmAxwGYzNYzEskFWTJLmW/p0VTRfiIk0d/6m7uhyx4pF1GSdtS65b7CYnvf7z0fs4NGLCcPC63O2IMyEd8ceVG/kIVKaY3nUkHJ/CkhShEcnVQ/vdq+uJj+iqKHv6PCgJcrXdAiI8KhIKKCYyE7aEf3A8pXkElXueaDSOPDSuNWBHhlXa/zi7V1bKhmRMkrtqyImZW+Ex4M9QgXU9ahDDbqVL8vq5rqjmks+Xd0lzmFE7KNehYkwcd51IW5aEWeqCWxgSxiPzOzNgPBY2EX+F2TFJFnSJsQtRzwb5g3wW/Q/11PaqeP4B814sQlVSlf0WntcqADgy8lT+AXn0WY9L/jdUORY0LwPnJczIYYVs/mpLNW+s4sk6YtKFFUCr7MJcNAknMHTrjg13LUlAlrxhP0ZTb+NCs7WVbUuK3Q0p6dCzzLszbVcLa5qRy0r603WSnhEiRVYr6BPYuUIwU21ui9bdB3xWjFKQ8KqZTDOgYBFj6EQ65lz3RnXpuFJlL/oaqyGWmzcQXPJKH6vzXoeWE4OOV7hUUUroB/Q+mljWlrickqgMoWA/qNfaCb4X3PoVC0dkZmWmXVguqkq/KrmThFLZFgadvDN2Cu6ANhOjTIcf4Nj06l93dyXDukHqbPDnqycpTL07SwLMCjE9aN4X7xvV9TPaIyDslilg3iem21lpMMUiDVHl883K+Lx9GA6fTbqP1oTQb8jzkDzUk69HqTFU8sKa2V6LDYjVDRSBZoNkLINda5ra9WoZGPpCnZlagYqtvCcoywPkrQ8bjM3yg3PJHZ3zVMJJ4JtZxRACmHqwjCBM2N7VefPuZ2M5GAu5N0plu61Yx618bUVFamVTCNbjdx6yCNUJkWtifW9Nj+x9421l0Ow3gKY+QIXIxGlDDtb10vr1/YOm0Fv6zppfGEHbA+cWwfc+tW6DowAvC06Mb60dxRxltLysi/xBnijvQNoGqjKcKMF1adnsYPj4etbli6LlYuwVDbR9mXG+WSDMYhmzvWx2GShUTM7CBQFGEXCU7wLMEDzm6oargXq+tDejXq+ZwPdetqHcqU6QzNfrz9Ro2lSz9OcIByna9ebn+IbrhPLOb7cv3y6HnDSuaSlb37V0w/EG++vr7trEKkEkLaOW/CpCjAoos/89pEhDJauzE82CYHehKIyjhyL+imjWKMmXTssiYZXVPY4sOtklfEjm30r387+wf7e5OXjby6qlBYMTG3kONcxulhsEpq4AskNZ4AGFS/ik0WK6NpRGtPw6d7WRb82LnElDczmxfDi+doYuZ6sOzGkA3zM1TOvbb/crYcyuN6OJgQ7wFicdJVbCPPsf58w5entV6BMI3IJ2qDnN8Cj+M/CRWcLZcK3wpGGv2wX24uDB7otFH6pzjFanaIb9HaZOFh8Hg20rIm52IMIQdcbXzacgq76aJgfiMff7YOvCOCqqr2eLC6oZgUf+l4v3AMUowgw6b+pCubI/7/6+PoQRYKXTPG4S5Sfs/V33UJCgw542zSgZlKH7FFQbDt74sxJzviusjwcviRqeSl6G8hef1Q+22yJZnmyCpCjlHMSgWJnybOMzajACXYYoLgeooeo502KVGr4HHbESTbc85uyrRSEZUaDJhwbQtha7rAUGSoIXZashVMSXAklvl97lafIcXXcNPKDHnDjCIKqSCYCkar/tpT/qa9PJcI7ItbUEWuheBoV1nyrEbGqOlQ+WN8xTSJdlCp/2z8x+YRCqoxnvTXXRXcce7MZ2d4WLWVETJk8oQZN2cpqtR6FWe8iTZI8oEPXZgSMRuludbdwxerSQ1VJYBmSa9QsbkQ54SS4uiwn6XK3vzcQTv5qcdbdVRpPfYUyBWo1utowAOxe8vEDYHKWKBWdxYDx1uJ7lVDYrlI5CQ3Aj1qpY3vYNL7LVr/5lmV4Uy1ltRgOTS0Oq1qtRrVTk3fxtb+n68QwlfpKqSZkp72hVHkMH/ym+SoXlsyUCUixW0DYW2DSFVMyiCinJSsmbG5NHbLt7d9gHTp28PvW0juqnKZdzyWotxDYWjyw1ZByn6qp90l+oT/amPJGWUQXyg+9Fs0KxVlv07O57UWce0hUf8Nk7HYl3b1smoC21tVtx83peI7JO8A/yxX2EtLRczrFH7CYvvvv75bfTS6++/W799+d/w/CpRazJb23qoLuBa0maD5BwVg1kmHnkufuNaweGrantXhOo8OeC5Nu09GOMUF0wXszVAjXsQyfbhg4emB+oNyi5A6/dwLDiyVc8IY6kjKkwtEcOSzl6HkGEuaJbbmq0DZJlGlRjmEa3yULvLeCsKiQknsn7Nh4rYADVXugXwvxXKHYCjukyzzlvC0Bwjz7dHfXzNS12Lhlmf0q7ksvrIDs2dnks8urnV/fGFiIJJg1CR2OjfANHe39+sZ7XU4JEZ9Mu82gmVODmVVTqr+68W1zxm1y8mliuagpu8uat8jlWJuzzHXClekIGqbrm8phyubopC5zZ1rUITVVXRyb32U+S2c3l/qFJIJzbe32tXYEroxIuq8i5wfIEyS3SRO1ncnBrvsDxrY8GNAAfyyOrQynGYKozE1zy6pRGtY/m9doyPFhkeBO9T9tl+zaMwEmHFXESZ1ZuWcP6JbYD2x8JeKxL+VImgmD9JjyXpuSk16Jqm3NolIaaka9brlUnBklFilz77yYWtXHZNaj+xZrjK8ls5HHHUmWlmu39VGQ8ag9abm4nQ5zxSeiG188otPntg9LBihNaztHtGpgRhXqFkWq9aIoZavd8LaZtrvS19/aKs+Oxu75c0nCtivW15zAa9rzdsl713ZF9mZo22Ws94m336BT14Zz3MDTbZssCdU0xW2aI9gwZVUhGRaWUvJKRwMob73peY2noPPKwL0a0Faoug9Cynby5FN0aVNM7QBVU9BKnR9L6tiS1dyq4mZXF2kax7dBaY2W6L4y8hiX0htR1F0ItVTeniRuSgL7bYaXzMs6bzjEe9fgXWVAa2ex8N26vu+tuXiyJR1bp2BVClG9DM3CwxvdhKZlb6O3XCZvE+y/np98eE1kW5vJ/c/lAJjqEwHVPeN6xF6zyn5LIXAb/RsVFijBRm7N9lreLWDEjk1AtJwYfN7dGpq5o28FF42/+0pT6lWRWQ9g+sHGiApiS9eVSkD1KzYqlPD/n/AQrCSrMwwMkSHp8bD2pllykag+s0v7TiDMtHvrIYmtZ2fud6Dm8Nu3ku625wRfnFr9ddKrbTT9lwv1Xy5Ui8fCuZzrmwa12z5tS6tZ40jMa8Dy0r8pfJe2iz6ack7e93HrEGS43WJbKp1udYaKAwjNYDkVQO3klrFDbvMA9YahCu349KilNejEauvzi9cnHy96zurPn93qP/4ZFVOMnR32XQ22MTsclbxLVMVQXXUvN9k36rMgoUsu+XJULBguL84wZ3fmqEqDxkcZbaGj40JDsDB+g3wyGdpSyMs/JcMI4UYfua8ZKE/fSjRuWZg3vRzN47g0KeJJR+F3b1dZCj1v35airPTosiO+aJ825sGm0yt9b6rGrwkmsBFmuO7QveTLb5EIMeIv2f7By8eH7TUsELWsYf1bK2qoPN66omF7VcNNKxtaqhuucyZIs0ctIe4wcLZjS5UlMx7xjIVezuIVU2ocVhZIr31trmPDrVnQiIW1V6McqpeLNAOOrAUsGyxg94rzz1hXD8SEkrhk+EOmWj7mvRKz/L2Rm2DerdKSAGbpt2GMiAs9yJfkGYUpJ+GKpzmbiwo5YjbrQtFYidKX5V61qpky7KOhsm7NUEoCGna2YHOPLeZh2rBqG6N88O/FuP7JzNZy9Xjz9NqjGjTmayOF2nbD63aEbQ3IEVX3FglohLM/4fFyGRzEABlgiPgOGiA3ZMUi45n/zyWvSq2vfqAfJVkW8Bu8L3HMrQWHqmsb6Qd6VR3BBj0Nta64YUQvGkS9AUcX1rDWKZjGeYdedy3lvumFqFvkg0iaODLh06udy++vrsC0/nPnzwPs8f8IuYI/u+Wrq6vsh8vh4Br/7ez++XJv9yefH/14/WPXPtlNOrVN2XVzSahDx90I8/XFzRW4OKZYgpfLseujVT7sPnZNvANYnms9yLo0cYh5pCAidwVS/2TuJ/vKtPuJJqiGj7MbtPZVayvmt6gZD8WVDA7by379IunVeE4LYIWSncoLjbvWlrJNcos5JMkI9KAbJnMZJZKWcunGp1lHCcuup7onlO5h31nvio6uq4tO5Nn1OR1OiftPAJFT2NuQGHYQSmlasBKWq9xeL00Wo0KPEN1ijTfhjT02S8NxEcHEYz3teZIns5St5hjxbRZY63nQlOM1rDF0MYNB49YKrA0SVgOj2n1ezMRDKnwZ2guwlSWqvhA58XkPy9n8/k3wk2WzvhA9+roHOk8Yhd8CO72S1xciqQLpeSvOYrC9vyqyqp8y5jMRnyQYlvaqEcg7vFo0Sm69fxSAfg7/wUpJVCRE4gKTjDlxHhbnh//cwnJLMZ2h5wkXuH1ZiKgNQ/hYKjo7v96uZv9KLlfo07GQfXIK1hnhHUe2tweDH8+DCZqcQ/rt0+7af27rj06ahIJ/y3EOso6ZY+mJF3T6P6wNAt+8s64CiQFkO+NiYtzwTbfbyuJtGGlHswj80ZRFPkZqW0K2ZjxGaZngzIiRvS2fdOrr60XHviyZhjmx9ixZR0hJOYyO3TcH+A3FP7b9zmDSofF/2yfVoIbVLyvkYhmE8ZSnpIWQoTTcf2YFWAA5qREYOnz4wre2ug0n+Xz4rG910s5pxm1vu75g/GZOkwipydgN6TLqMiLjr67bR1MSaBYgPgGzz1+ZqaPA09WEqHoxTT7OtC+uk6ZIE83H1Viqde0gY52WpZ4EXj9IJEyTGrSMBsjqBK7JwpvcUXG0RJtjJgryCRBZ2F7eX7W5azFaHx5btvC7fJ5QMBlGscgKvs0NYbYqsA2WibPXQ75J2TKYjSgjh/yE8gG6BdvqAad8qtdKPjt687XqB1tKE39B6eIvyLNRjp/UqCyj6GHvCwuN67X5B96aQg7WMgRYoUQ9GFQudXPVyFabK5dvOFrXLI/f2D2I9i8NL5PIhGo+b6H5iqewmSxF1XAzQaX5XdNoLYfpCvcWZwr2oPTqyMzSUbgKpinnv9MEK0co5LiR59zWwDMjwgxjyZbClFhRPRNRdocAX28XZ6XFiWxaWtt2HFNGBGxVX7v5fUu+2OZlE7UwfoW1e7rCpt5kaGor8taMttxG47C3vGbDRHvNAe9XWM9idennwci+NvfnYxe3o+69+r1G4zVF8P8lPjYXHy2H/4883e+2UGOrVNja/2JlhoZ7xvG94jCpNk/V2eVaAuI7PgmEg2WgesgwBs6itjQttgGIW5y8xiU+jbYWeE2B5iDEBiL08brIRjJ8iwAZRXKb8vHLLkdA8pYLUxQPcCS1VClY7hsLm+y1pnFrFNOab9dP36YADPK3fmZuWL22W++0TbAkcs9ImSufN611GUCDB2Et962VzY6UXCVBBvpoe6Zvq5VTkqoVtMnOFoCnqOxnmK6GHm9RlFwhi3FYD3/Rwbl2uo5vdhY8jUGGwiZbPqI63KuI3ZGVhEf0p2RfeU/LI3o8yCtAeOA7YXuVb8ovxNOnO3WkQPURWvhJ3Zf+RQlHmjIUH+D3+zvlIAjWTjzCvY1hOMGB9iBYhnGSwuNn//5v0Pz/Ajkea3H61wAA',
 '03_evaluate_public_multicrop_lora_colab.ipynb': 'H4sIAAAAAAAC/8U8aW8bObLfF9j/wHXwMNKs1I4cO4cwGsATO5ns5DAST/YBttFLdVMSo762D9saj//7VvHok5Q6FyYIZIlkFYusu8juu7//jex5LAiyvSm5gB/kDj9km5tvEgbteyFN1358E+2NZCf3sfXp44f+E/8R1a0hy6lPcwp9d/eqLYuL1GMaNzQ8IOc04sc+TXK+IGNyek2DguaMvCmCnMvh8ruXxgl5Hb8/vtTzkr3a17dxzuZxvCYhC+c08nm0XNOIzGnGyIeczgNGTvhiUWQ8jsjEOSI+9CI64rNoCd+TNA6TfFRDmTHmj0jG0pyShKYUFsRSwmF8ztcOOWMRDygH0DULaCYQZus4JSGNYBoKkID4E61h9Pmcw94hSJzF0ZJkNEwoJym75uwGkIcs4CuakyWFRaQOgArIK/yUe2jihxf7TO86u2VekcMiXS8uohy6oyIIGow6ZN4Tf37waBuj4iJPilxIwZWNdzxM4jQnXnZdW6Jq/ATL67bGWbctK+aw9R7LTH0bQ2POQ1ZrXQDfiBcHAfNw2RlRw57j8lnaHgkLZYhBD9O/RwLvH3HUwZ3QfBXwuQY4g59mEQQBvtx7/u718S/u+9PXp8cfTt3z45eXeySKc8IjWL3DomuextG0BoUfKeUoppssZ+HpLc8Hl3tSETIO09IlI0sWsZTiAsmKRhtKfP6JBjRCWfI5eRnHS5Dw53FA587l3tBCoNrAOPXsS0BixQjHK3zq8Myl1xQEHTRoMNxNeLMf/8Ge/H5yDNvr0zUBjmTM59QBbSQfPwDFPpuSciliAVPynoXxNeBl6TVLR4DBiDYv8iIRajcv1hS0EHQm4LgAYQlGJIAtJAkHlQJUKB/kN5ZGLCA/y4ngrwX1W3ajhkgaYOTLs9/h8/zQIf+i0mAIxMdFHsPQKEL0oEQF0CHmCOI1DZwO/jprKtF30iIaXFzugXjA5oyzkF/uXY2It2LeevaCBhkbfh9b8NA/OvSfLY6+1hYITVkKKXQ8sXFav1J+zczCJrqcEEkDgd/3YtDXKN+XEHvfacHzx95k8Yh9tfE7ef/q46n7/t27czITNqGzhP03mxPxN1unPMm4G8Qpberm83dvX7x66Z68eg9Iahj3QQSDeJntI2UZy90kZeiBcGk4yYIvs4Zkpey/BU+Z76pOQNdWxdpciJ5JR4t7lRTzgHtuqL2ss6FhcLk32o4gB92PwMv2AR82TUzIswwgyXRGLrI8HaCFHZIFuE78hqaysx5lmLAfTdKCozW6+jJz9JvAWaRoXZk2s3MWFGFpntCmCuY5NgvxL22ytLkhNR4hnoDNwTz4dFUEhUPeyCVPAZL8Ez7QqDmfYh4N1GYMt9gJu8loAV0YSAUf6ki9QBM+Mi5mHHbZLXsSnti6eJTlNAhs3ePfbT2+iMNgp2ezh86jI2diGwgiFmUgFiGM/Xl26BxOnIPRT0e24dQDw4Bekv2MiCfOw9FPB9aFsUWOwyZPcJiVhFWxXAJ3FtRj41UxF4gPt0JkdAFxYpTFSDOMdibGwVftBmnrz9OC2TRnB+9bjNZsLZkIX4qoYhr2jzcsk1+Fx6cxep3edH0H63z0ZMHmk8fPvlFoKixROwDU0qc9lMwOyuTgjCcQhHfDwLNXrzXEKwzJRvLPSUpv1NcXPMgxXBE/AGveRlETDY0Km9w4YVF7bF32y5D29auzN7ChwUh8PZPCEKdmU3H68fj178fnr969dZXpPjs+/xU8w2d7ghrS2mBpmgEfDnLEQoKY+gPzvE7KqO/m7DYfsAikApRqBgKZL8ZPwSfWxbx0LHICV/iEFtXC4XZoubhsA4M8D52INhKG1pDuAkwE9Ce/FqnPiInEagBQZ2Hd/5+dPj8/PXHP3796+fL0/QdAddfSS1Br6kOYiC4lhxyayhzala2jzuhPdFlEy+543d6F8OicGiZQzd3xa+pBZOzCWLrqgjV769D3te+e8s3g+vOUL5co/bB2Dg4dto77sGVTon6pAW4erxnupYggsA8jCNPOe5CmZ2jxru6bAYlp1n/MSIcNPfK2cwlfBRYy74FpC0j017XQY8EjkSBYPL00c24aQ9gzswu8HAZLQskS8bcrok/Xh/jJA7O+QR0w43XCNYyC8CuFoDWTNp6wW57lbixNfh0Q2BdH3KOBCGRFCIaaBcS1FKZFVWN0U+Rh69tuTOTKmXBTkccGlklHBOLGdsgE3EdoCwjk3n52w0Wg7tQscZMBCo/YbgsiaVBQPiwD6m6yh8S8pSElCCvLUSVSCHZ9qkSz3GIlTiAI3G/JjqBBGcv2ror1bGdTL+HBj/0+6zbLdD35Ywsd5btiNQKXTAXam6YWJRYhBrT6jcnB1BCd9ckP8N/icq/FC5ke+GhdCoz6MzanS8oF3yA2wanvLbmCLV8oWapyBCOwYalimaAe+WAIf9yM/8HIT2Ty8OCw15K7a1O1SFG8wMQFqyZr5vFAL6yrInm6MU2G6lWFNIJXI7LAyulNnK7BYSY5hps+u+Yeg59eUgBuQjMsavmwk5btW7MNuoEALNNAjnSwaTAcWgBgaSxM8g0AZUWoYZaQSUutHwA46HERsmAwJD+Th8J3QCO6DkRtQ6wDUhEKAHasczp+ESaZTZbwn5pfA8OUMBsEtARIyV2cTxlf0AoKwdcMrJsF3dDxaMYWceAP2jSyW48lOTkVfzD+gH2Fti+UirmQ8w/AzHMVsgo1BZkAnCgSRMSp8KMromAPBmIXQTCfSrus+AG/+ylmmxxp90JQP5oTyUQiQgvwtRsILIhXrIvE6YoqkEOjjYk7eIyBBf2oyVXDyIWo6uvRQFwSQEK4AiawVOZPIALhhtAUIh3q5bIt20T5ikEkXWvvpPlfuBss9cGBrSGM0JaoRpJhF1KWF2kkbMdO01x3JAYTXXZbjLw8SJE5ntI/lAcVtMlejMGGnxGvtQpIjSnA9h30LqgLvVA+dUXTIkOR4gFfcxKCNQ5hbycHCn9rG5OUi0LlR9wKiUgtZor8rtP0nbLjZ4cHj9hk8uRrs+NEpbjAHUvy66BuY9VROCnmt9XHxC08anNDTE8NVQRRX3B9XPhMni4sINfKJ4/bA0VktnFF1YGls7dgOGz1kNoy9Fcnj4FDeHBxaRwKQSCu16U51mmR/AwSXnDCg+/EsqcefeqxR4dfyzIIx+LURyd4cdUrtlJZJmpuyikIrvo70lZT5LrM7xggNQ5DQJitnoXs6z47xOdlFNq0Cj13+1kCk71U8D7QW6HS6aJ1vFxgfbn7JaZtQJ+/Rr1OZSJ45LNbbTDEijEQQWYNGuRXphLTHfAhs8nQFiEhejyvRnRVgQGQYKNt3/Q/BRDjfkjlfKlbBlWsJpXKwdNtiPwR7wANIn6xxmFCo5F0yKwpegM8ZHXwY7ANRJ55Vmq9LbYSCYLYqVn3CN80OGJL2BxIchRUY7tanQZL1kFXhMDRBQNp8JgLDifJZrgvDbStAejRduFdFuBlBAQ4fDYT5rKJtDmiD84b7uerLnWiuQ/8ivHlKu8ikO29VqUFa1Z+2wYydIQoZBcPr7aMSsH4gkiBTILBEEYyBXvsD2rCRsY1ORyRRzulz1W1DQiM77RVuHfvlLm4V/JxV1fp6cMDGIL6cIcf906CdbWdE6lIqzJK+zUKdkE7Gb1mgwrRtnUpH+LQBJIzf5dO3e3ol2ktu02Yh7sqyMeqn96sUS947VEAUDuoXnBNJ4bgzZZ+WLSST0k/0yGhOjZiSr7ShqiTIyZXUlrVfmAd2yIxfK39kcjbNmZKvokhksiV3THQ29siqVM6ZX8MmPqbJolKBTwiUQJ8eDxd065+OFr2CPG0mvrhQQiwWmGCGPRFJSeKbwb6rpJT5B7YyCzGkyKsCO1CfL/N2O4MLDtBpYz6Mf0pL6l0Qm5USFkovBG86FRKcI/LpNKRcVVnIRJW2MRZNVjUgi3pgYFYeQNjRHSQ9j3ulzx9+Hgxp/7XBvwQZvEFcF+7h2Yg3ji100Mdcemuzq8GEucmhcBSnmHVylbKK4wI+rAonx0MIZrtHnGZsHrZtY08kQxeZ5LvJYF4VbBOn6gXdtA5ongIRgBZFbEblCKgBH8ZCLNXD8V6MaYFxM4J9/J/iwZVDRyRBWeBjxKUzURlUe0ExBqdeFaikju4YuBg0sG2IWl8U26sRauevz7+8MF9ffzL6etdp3uUJKs4j5cpTVYkXpAUAnIs90D2QZYwFdaQIeynckG7zv7a2EKKBWTMwmIsEH0O2toBYRurt+IBJwmDUCP9PKTdU8Q27oRBCpL3xdo4XQx4IksUMlsEcw+iRvk+doyveT5GuR2DEHqrRwcNURWgiT5xx/Po+gl8p17SmGloJEHhEAf6O+BtZY1qlIP2YNDv2EX5szS+3bhgnkCctLCOSCivF5QL7SScQmXFDQepphLwouU4r4Ra4gDmmzJPneDJEQ74xGuWooV+//KXbvFSJA4pxWq+DHqr8a/to108I8FkoLyP4eDHAPtMMMxfsm0wzkLc7xjU7no4L169PXFPT16efjCmwDxCO4+JrN5PW9SNVllaobphcNDIMzznsPl1mR7NxB/bGFn7VScfmT6FsQwGsyMMbPvCj/VASi/xbs02GMQDwQ1x1ScrI9mHeiphHCxzwNruTRImyxBR7MLO++bDPNjTOZ2Dmck5QwKE6A5+/FFiH0LcseR55oIBclWqFC/ykN4OfB7OJkMw846XFANULrHx7ZVlINgCsQ+uY/AHTxqsGTXnFwfSME7WfzrVdxqtGRocPO8B7Zao9QaMcHtmAQ3nPhX1cHXDYXI1wicD8EqisawE5gJJ08nXSNa1fQzzMQUWc3YTZ3UScGe8vNZN50r9bnddma+/1RK6ErRss8BUeVQJsjV1qlKlcrxssIxuxfQWk2WGrUysTG5rZnkLQIs1MsNscssO3NxpVwiLoFsUNaTsXNj5AlLzeLgFfR4nbiUpFeKqrQeCkKZLHpmAyViL3uRKiLAdF64OVoFXDaQ7MuwTmc0+WwbnKeYKERhc8G+0RmXlFhzsAOUYkUMbeXEEzhgP2HPfZ9dGJLJrKxrlVXBctUaJqHI42xHd7/TqtMhjSP9gYzDyxEOCbT6+EmL1vfL2wljLcfIatghhr4xTKbS2DKAz0Bz9m/H9VSlAcyO/VSbQxGoJ0lCDxBmczC2Ri+oRpnbMMAAsTatKZEtXPVSH2SapE1gYIoLnBpGWnLqaoU1jf6K2zDo07oeW6dY23O3yP1sKiiZP12yxearGHlqtls2kFJIy8cXi2VY0ZS6qBo9KT1C3s3jkvU9ajLiojveayxhuMcBS4Xq6NXwOrORVuWxTlHhvOOGzkTdqb9xwJDcHZUMFS23F0GFTXViujIIjzZ/FPmn30xhstlEmjH+VhWqqxbeyUE2s1mdPQnD7m1If65uOTFY8VldkdGFQV906MXwmHskTYfEFWgOrVcArPG3rgmGB+tUOcOtkbjnruLNe1et/JgFqIfIvHIp3XPSShnYAKXbU84qUeptKk/CajDbWhrjoqr495TSWWfb7EoNBkDXcrFNkjUq/J12myKkiytD9rYm57/dMVq0Uq2Vvd9FWjXRC/9KkY2hOTE7vcu8Bqd1k+iCHm+poprZfacbxLmvK10WOJ2YBXRGxeSQWGsczUZPax6gUrA1fqyfRR+o6YFI+837Ns4IG+ll3py8Ff5KPykr8KWssGXx5LgJ9rROKoD/JG5CC0l+MZTYgxK7sAxHQo82TjcdjIj6nlj9NsLZF06JUsyhTi7URDNtibhZIzh2K7g/KqPxwdU90kzQj9RaTnfjhauocLnCM8VgZj6oFrFWtKwT1gW1FKkfteHi5vnB2mxsWfmEMJWxPzj14QH5joC9FBrx9I1Pu0WdhGON9NoCfoqCiCyarIqyEtVI/x46hfNnE/oIGwZx662l5+1sQJx65Bxa2FSejn+gWvC8UNoIVcfnGCI1WP/7u9F/vlS1AN9ui+tkPoLuM9NOnDS7uPPnBqnWtIFJ7pqp6YYC45+t61FsxWz1aMAU8HKw+Ku91105fyD8Ndk/y0P0vfHCYYOIeGY8kgHGufEWHfZC6tEH9lToUt4yjS4iPQRxABQM3BU9xTbeM1i/xcPGGf2Y3LHL1Fg8hO2tchFg3CSAUMkemFa6/IB5tsPELolBjrcH4WEEzXFWi0iOP1gJmvFJYhbyCcfIpGAVzacso+uUSEuPOZOKvYVtng74n6zB075aB8eEpnRfYLiZi/Y1HBeshDxYM224wfX2p21QUUDaqxGjzVF9e+v6yEnjtaoo2UmWYv5vGRkLSDyTkt7h9aIzxOQVdw9gNqGzn9qH3lnOp73Cdgx5NDg8XT+nXXufAc9eQrplbmoqcQnaXrRjL9a1f+11sYLBQ7judVk/JwyqznpJJe0ckxurSuCqlGPZNGVOzRjfUt6MVmIOXpJt4VTvWklimZGA9o+wr/X3lvbF3FwYjZITsrKPz5BsPsDznCz9wcPTY1C3vnGH/s3Z/5ZMO2udysqoinpVpsA/DIQ02JpMh2d/Xv9u5CEqTPrx2wJuY338izthNix/oaX6sLXMkKfuxvjRLQfFmBQa5i7q9hX5KbzSV+PoEBz8GgniTn1HX5SvhbF+X1/u07aJBjxsLfW4hyIPkHCKfeUR5MBjU9wmkwXgL4BbZKlZB/k9zbtjYYwPQpgKq+D1sssH0thfcRSehWa5uBY/IADZvYyQsoHNxDUUkjnJXmsmpeKKg7MJf0G7MPZGnjkgvBrcgr4cwJfw5eIy3ycQ0GIIEwWwAdgv/d+jJaboUwtuMrRbyIK60lW51Gdt4vVquX1yKlhgtD8LJzp0Ha1/7dIp+Zuy5XIMkTzwutssZ1J5gGVoftJPI36gLdApv7cqf6fG1X/CG03U2Fk8UPv/wsQml7+CZII91VZaoozoBaj7AM8HLMk8Zfsr8WeAwFdhNGGqVL5W4CnBL0mvC8KuoB6TFHCgVsFUGZSS5JFYGTm1qFVifwMP80szD+TP/8fzp5PNemvmAvBfkkDfiYRyzeLzCh0PxfZST8ZF6aWAGsRiW/GRRj/hcvG3SIWf4poH/2LOY/9RNOge8sn6hyiri8ds5i2g6Fp/iIXJEDMKR0DXMJ4erd2Coq/b44r76ubIc6ReUrOMQEp8cIMQj6fJFWoDTo2FSpOX790IWLZEWUZEsyzCtd2bCh4jJGhuLPXvyNX0ZxDS6Cd/PmUGyvRG3jJE7Z5t8BbwvXxYXwMwFGFXsS0Sf7tEQsvXRXiULJRA+JRRXczUhNB687QN7jx2PnMlkTy9C4NqL5vKeN3QfNhrckEdxCs1Hf/8bDP8fj2/1isdVAAA='}
EMBEDDED_NOTEBOOK_SHA256 = {'01_prepare_public_multicrop_dataset_colab.ipynb': '3f042dec35d38a58054ddfba83ea04e0bc61966be49bafa8d5d1d9a362ce4c64',
 '02_train_public_multicrop_lora_sd15_colab.ipynb': '60b194711ddd2aa9eb2c3e0a73c215637db3cadea076ece081521cc0a407639f',
 '03_evaluate_public_multicrop_lora_colab.ipynb': '07d6c3dff9e61fbf081ba29c5c08f059623751d0bf16f32613e6cbf3110ce2e6'}

def materialize_embedded_notebook(filename):
    try:
        compressed = base64.b64decode(EMBEDDED_NOTEBOOKS_GZIP_BASE64[filename], validate=True)
        payload = gzip.decompress(compressed)
        expected = EMBEDDED_NOTEBOOK_SHA256[filename]
    except (KeyError, ValueError, OSError) as exc:
        raise RuntimeError(f"Payload notebook tahap invalid: {filename}: {exc}") from exc
    actual = hashlib.sha256(payload).hexdigest()
    if actual != expected:
        raise RuntimeError(
            f"Checksum notebook tahap tidak cocok: {filename}; expected={expected}, actual={actual}"
        )
    notebook = nbformat.reads(payload.decode("utf-8"), as_version=4)
    nbformat.validate(notebook)
    target = RUNTIME_NOTEBOOKS_DIR / filename
    temporary = target.with_suffix(target.suffix + ".tmp")
    temporary.write_bytes(payload)
    temporary.replace(target)
    return target

PREP_NOTEBOOK = materialize_embedded_notebook(
    "01_prepare_public_multicrop_dataset_colab.ipynb"
)
TRAIN_NOTEBOOK = materialize_embedded_notebook(
    "02_train_public_multicrop_lora_sd15_colab.ipynb"
)
EVAL_NOTEBOOK = materialize_embedded_notebook(
    "03_evaluate_public_multicrop_lora_colab.ipynb"
)
print("Runtime stage notebooks validated:", RUNTIME_NOTEBOOKS_DIR)

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

if PIPELINE_STATE_PATH.is_file():
    try:
        pipeline_state = json.loads(PIPELINE_STATE_PATH.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        pipeline_state = {}
else:
    pipeline_state = {}
pipeline_state.setdefault("schema_version", 1)
pipeline_state.setdefault("history", [])
pipeline_state["pipeline_revision"] = PIPELINE_REVISION
pipeline_state["dataset_sha256"] = EXPECTED_SHA256
pipeline_state["gpu"] = {"name": gpu_name, "vram_gb": round(vram_gb, 2)}

def record_stage(stage, status, details=None):
    event = {"stage": stage, "status": status, "timestamp": utc_now()}
    if details is not None:
        event["details"] = details
    pipeline_state["current_stage"] = stage
    pipeline_state["status"] = status
    pipeline_state["updated_at"] = event["timestamp"]
    pipeline_state["history"].append(event)
    PIPELINE_STATE_PATH.write_text(json.dumps(pipeline_state, indent=2), encoding="utf-8")
    print(f"[{stage}] {status}")

def load_and_validate_notebook(path):
    notebook = nbformat.read(path, as_version=4)
    nbformat.validate(notebook)
    return notebook

def execute_notebook(path, stage, environment=None):
    notebook = load_and_validate_notebook(path)
    environment = environment or {}
    if any(not isinstance(key, str) or not isinstance(value, str) for key, value in environment.items()):
        raise TypeError("Notebook environment overrides must be string pairs.")
    previous_environment = {key: os.environ.get(key) for key in environment}
    namespace = {"__name__": "__main__"}
    try:
        os.environ.update(environment)
        for index, cell in enumerate(notebook.cells):
            if cell.cell_type != "code" or not cell.source.strip():
                continue
            exec(compile(cell.source, f"{path.name}:cell-{index}", "exec"), namespace)
    finally:
        for key, previous_value in previous_environment.items():
            if previous_value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = previous_value
    record_stage(stage, "completed", {"notebook": str(path)})

def assert_final_config():
    config_dir = LOGS_DIR / "dataset_preparation/configs"
    training_path = config_dir / "training_public_multicrop.yaml"
    evaluation_path = config_dir / "evaluation_public_multicrop.yaml"
    if not training_path.is_file() or not evaluation_path.is_file():
        raise RuntimeError("Config final belum diekstrak oleh notebook preparation.")
    training = yaml.safe_load(training_path.read_text(encoding="utf-8"))
    evaluation = yaml.safe_load(evaluation_path.read_text(encoding="utf-8"))
    expected_training = {
        "resolution": 512,
        "seed": 42,
        "rank": 8,
        "train_batch_size": 1,
        "gradient_accumulation_steps": 4,
        "max_train_steps": 1600,
        "checkpointing_steps": 200,
        "mixed_precision": "fp16",
        "gradient_checkpointing": True,
    }
    mismatches = {key: (training["training"].get(key), value) for key, value in expected_training.items() if training["training"].get(key) != value}
    smoke = training.get("smoke_test", {})
    if smoke.get("max_train_steps") != 100 or smoke.get("checkpointing_steps") != 50:
        mismatches["smoke_test"] = (smoke, {"max_train_steps": 100, "checkpointing_steps": 50})
    if training.get("base_model") != "stable-diffusion-v1-5/stable-diffusion-v1-5":
        mismatches["base_model"] = training.get("base_model")
    if evaluation.get("generation", {}).get("seeds") != [42, 123, 2026]:
        mismatches["evaluation_seeds"] = evaluation.get("generation", {}).get("seeds")
    if sum(len(item.get("prompts", [])) for item in evaluation.get("classes", [])) != 12:
        mismatches["evaluation_prompts"] = "expected 12 prompts"
    filename_contract = {
        "smoke": smoke.get("lora_filename"),
        "main": training.get("output", {}).get("lora_filename"),
    }
    for run_mode, filename in filename_contract.items():
        if (
            not isinstance(filename, str)
            or not filename.endswith(".safetensors")
            or Path(filename).name != filename
        ):
            mismatches[f"{run_mode}_lora_filename"] = filename
    if len(set(filename_contract.values())) != len(filename_contract):
        mismatches["lora_filenames"] = "smoke and main canonical filenames must differ"
    if mismatches:
        raise RuntimeError(f"Konfigurasi final tidak sesuai: {mismatches}")
    return training, evaluation


In [ ]:
MIN_LORA_FILE_BYTES = 1024
MIN_LORA_TENSOR_COUNT = 8
PLACEHOLDER_MARKERS = ("placeholder", "dummy artifact", "synthetic artifact")


def inspect_lora_safetensors(path):
    path = Path(path)
    errors = []
    tensor_count = 0
    nonempty_tensor_count = 0
    metadata = {}
    size_bytes = 0

    if path.exists() and path.is_dir():
        errors.append("path is a directory")
    elif not path.is_file():
        errors.append("file does not exist")
    else:
        size_bytes = path.stat().st_size
        if size_bytes <= 0:
            errors.append("file is empty")
        elif size_bytes < MIN_LORA_FILE_BYTES:
            errors.append(
                f"file is too small for a real LoRA adapter ({size_bytes} bytes)"
            )

    if not errors:
        try:
            with safe_open(path, framework="pt", device="cpu") as handle:
                keys = list(handle.keys())
                metadata = handle.metadata() or {}
                tensor_count = len(keys)
                for key in keys:
                    if handle.get_tensor(key).numel() > 0:
                        nonempty_tensor_count += 1
                if not keys:
                    errors.append("SafeTensors contains no tensors")
                elif nonempty_tensor_count == 0:
                    errors.append("SafeTensors contains no tensor data")
                elif nonempty_tensor_count < MIN_LORA_TENSOR_COUNT:
                    errors.append(
                        "SafeTensors has too few nonempty tensors for the configured LoRA "
                        f"adapter ({nonempty_tensor_count})"
                    )
                metadata_text = json.dumps(metadata, sort_keys=True, default=str).casefold()
                if any(marker in metadata_text for marker in PLACEHOLDER_MARKERS):
                    errors.append("SafeTensors metadata identifies a placeholder artifact")
        except Exception as exc:
            errors.append(f"SafeTensors cannot be opened: {exc}")

    return {
        "valid": not errors,
        "path": str(path),
        "size_bytes": size_bytes,
        "tensor_count": tensor_count,
        "nonempty_tensor_count": nonempty_tensor_count,
        "metadata": metadata,
        "errors": errors,
    }


def inspect_training_run(output_dir, target_steps, lora_filename):
    output_dir = Path(output_dir)
    errors = []
    manifest_path = output_dir / "training_manifest.json"
    manifest = {}
    if manifest_path.is_file():
        try:
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError) as exc:
            errors.append(f"manifest invalid: {exc}")
    else:
        errors.append("training_manifest.json missing")
    if manifest.get("status") != "success":
        errors.append(f"manifest status is {manifest.get('status')!r}")
    if manifest.get("target_steps") != target_steps:
        errors.append(f"target steps mismatch: {manifest.get('target_steps')!r}")

    checkpoint = output_dir / f"checkpoint-{target_steps}"
    if not checkpoint.is_dir() or not any(
        path.is_file() and path.stat().st_size > 0 for path in checkpoint.rglob("*")
    ):
        errors.append(f"checkpoint-{target_steps} missing or empty")

    lora_path = output_dir / lora_filename
    lora_validation = inspect_lora_safetensors(lora_path)
    if not lora_validation["valid"]:
        errors.append(f"canonical LoRA invalid: {lora_validation['errors']}")
    manifest_lora_path = manifest.get("lora_path")
    if manifest.get("status") == "success" and (
        not manifest_lora_path
        or Path(str(manifest_lora_path)).resolve() != lora_path.resolve()
    ):
        errors.append(
            f"manifest LoRA path does not match canonical filename: {manifest_lora_path!r}"
        )

    loss_path = output_dir / "loss_log.csv"
    loss_values = []
    if loss_path.is_file():
        try:
            with loss_path.open(newline="", encoding="utf-8") as handle:
                loss_values = [float(row["loss"]) for row in csv.DictReader(handle)]
            if not loss_values or any(not math.isfinite(value) for value in loss_values):
                errors.append("loss is empty or non-finite")
        except (ValueError, KeyError, OSError) as exc:
            errors.append(f"loss log invalid: {exc}")
    else:
        errors.append("loss_log.csv missing")

    expected_classes = {"padi", "jagung", "cabai", "kacang_tanah"}
    validation_paths = sorted((output_dir / "validation_images").glob("*_seed_*.png"))
    observed_classes = {path.name.split("_seed_", 1)[0] for path in validation_paths}
    if observed_classes != expected_classes:
        errors.append(f"validation classes mismatch: {sorted(observed_classes)}")
    for image_path in validation_paths:
        try:
            if not image_path.is_file() or image_path.stat().st_size <= 0:
                raise ValueError("not a nonempty regular file")
            with Image.open(image_path) as image:
                image.verify()
        except Exception as exc:
            errors.append(f"invalid validation image {image_path.name}: {exc}")

    environment_path = output_dir / "training_environment_report.json"
    environment = {}
    if environment_path.is_file():
        try:
            environment = json.loads(environment_path.read_text(encoding="utf-8"))
        except (json.JSONDecodeError, OSError) as exc:
            errors.append(f"environment report invalid: {exc}")
    else:
        errors.append("training environment report missing")
    if not environment.get("gpu") or environment.get("target_steps") != target_steps:
        errors.append("GPU environment or target steps missing")

    declared_log = manifest.get("training_log")
    training_log_path = Path(str(declared_log)) if declared_log else output_dir / "training.log"
    log_tail = []
    if not training_log_path.is_file() or training_log_path.stat().st_size <= 0:
        errors.append("training log missing or empty")
    else:
        log_text = training_log_path.read_text(encoding="utf-8", errors="replace")
        log_tail = log_text.splitlines()[-30:]
        fatal_patterns = (
            r"Traceback \(most recent call last\)",
            r"CUDA out of memory",
            r"(?:loss|train_loss)[=:]\s*(?:nan|inf)",
        )
        if any(re.search(pattern, log_text, flags=re.IGNORECASE) for pattern in fatal_patterns):
            errors.append("fatal training-log signal found")

    training_failure = {
        "manifest_status": manifest.get("status"),
        "return_code": manifest.get("return_code"),
        "error": manifest.get("error"),
        "training_log": str(training_log_path),
        "training_log_tail": manifest.get("training_log_tail") or log_tail,
    }
    return {
        "valid": not errors,
        "lora_valid": lora_validation["valid"],
        "target_steps": target_steps,
        "dataset_sha256": EXPECTED_SHA256,
        "manifest": str(manifest_path),
        "checkpoint": str(checkpoint),
        "canonical_lora_filename": lora_filename,
        "lora_path": str(lora_path),
        "lora_size_bytes": lora_validation["size_bytes"],
        "tensor_count": lora_validation["tensor_count"],
        "lora_validation": lora_validation,
        "loss_observations": len(loss_values),
        "final_loss": loss_values[-1] if loss_values else None,
        "validation_images": [str(path) for path in validation_paths],
        "gpu": environment.get("gpu"),
        "training_failure": training_failure,
        "errors": errors,
        "validated_at": utc_now(),
    }


def require_training_run(output_dir, target_steps, lora_filename, report_name):
    report = inspect_training_run(output_dir, target_steps, lora_filename)
    report_path = Path(output_dir) / report_name
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    if not report["valid"]:
        context = {
            "errors": report["errors"],
            "training_failure": report["training_failure"],
        }
        raise RuntimeError("Training validation failed: " + json.dumps(context, default=str))
    return report


In [ ]:
def read_csv_rows(path):
    with Path(path).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

def inspect_evaluation(output_dir):
    output_dir = Path(output_dir)
    errors = []
    manifest_path = output_dir / "base_vs_lora_manifest.csv"
    metrics_path = output_dir / "automated_metrics.csv"
    confusion_path = output_dir / "class_confusion_proxy.csv"
    human_path = output_dir / "human_evaluation_template.csv"
    review_path = output_dir / "class_confusion_review.csv"
    required = (manifest_path, metrics_path, confusion_path, human_path, review_path, output_dir / "evaluation_summary.md")
    if any(not path.is_file() for path in required):
        errors.append("one or more evaluation reports are missing")
        return {"valid": False, "errors": errors, "validated_at": utc_now()}

    manifest_rows = read_csv_rows(manifest_path)
    metrics_rows = read_csv_rows(metrics_path)
    confusion_rows = read_csv_rows(confusion_path)
    human_rows = read_csv_rows(human_path)
    review_rows = read_csv_rows(review_path)
    if len(manifest_rows) != 72 or Counter(row.get("variant") for row in manifest_rows) != {"base": 36, "lora": 36}:
        errors.append("base-vs-LoRA manifest must contain 36 base and 36 LoRA images")
    if len(metrics_rows) != 72:
        errors.append("automated metrics must contain 72 rows")
    if not confusion_rows:
        errors.append("class confusion proxy is empty")
    expected_classes = {"padi", "jagung", "cabai", "kacang_tanah"}
    class_variant_counts = Counter((row.get("variant"), row.get("expected_class")) for row in manifest_rows)
    if any(class_variant_counts[(variant, class_id)] != 9 for variant in ("base", "lora") for class_id in expected_classes):
        errors.append("each class/variant must contain 3 prompts x 3 seeds")
    for row in manifest_rows:
        image_path = Path(row.get("output_path", ""))
        try:
            with Image.open(image_path) as image:
                image.verify()
        except Exception as exc:
            errors.append(f"invalid generated image {image_path}: {exc}")
            break
    human_fields = ("visual_quality_1_5", "crop_identity_1_5", "prompt_adherence_1_5", "agricultural_relevance_1_5", "reviewer_notes")
    if len(human_rows) != 72 or any(row.get(field, "").strip() for row in human_rows for field in human_fields):
        errors.append("human evaluation template must contain 72 blank assessment rows")
    review_fields = ("predicted_visual_class", "reviewer_score", "class_correct", "mixed_crop_detected", "notes")
    if len(review_rows) != 36 or any(row.get(field, "").strip() for row in review_rows for field in review_fields):
        errors.append("class confusion review must contain 36 blank human-review rows")
    contact_sheets = []
    for class_id in sorted(expected_classes):
        path = output_dir / f"contact_sheet_{class_id}.png"
        try:
            with Image.open(path) as image:
                image.verify()
            contact_sheets.append(str(path))
        except Exception as exc:
            errors.append(f"invalid contact sheet {class_id}: {exc}")
    return {
        "valid": not errors,
        "generated_images": len(manifest_rows),
        "automated_metric_rows": len(metrics_rows),
        "confusion_proxy_rows": len(confusion_rows),
        "human_template_rows": len(human_rows),
        "contact_sheets": contact_sheets,
        "errors": errors,
        "validated_at": utc_now(),
    }

def require_evaluation(output_dir):
    report = inspect_evaluation(output_dir)
    report_path = Path(output_dir) / "orchestrator_evaluation_validation.json"
    report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
    if not report["valid"]:
        raise RuntimeError(f"Evaluation validation failed: {report['errors']}")
    return report


In [ ]:
from pathlib import Path
import json
import shutil
from PIL import Image

DATASET_RUNTIME_ROOT = Path(
    "/content/taniadaptif_dataset/multicrop_v1"
)

def repair_runtime_metadata_paths():
    report = {}

    for split in ["train", "validation"]:
        split_root = DATASET_RUNTIME_ROOT / split
        metadata_path = split_root / "metadata.jsonl"

        if not split_root.exists():
            raise RuntimeError(
                f"Folder split tidak ditemukan: {split_root}"
            )

        if not metadata_path.exists():
            raise RuntimeError(
                f"metadata.jsonl tidak ditemukan: {metadata_path}"
            )

        # Backup metadata runtime sebelum diperbaiki
        backup_path = split_root / "metadata.before_path_fix.jsonl"

        if not backup_path.exists():
            shutil.copy2(metadata_path, backup_path)

        rows = [
            json.loads(line)
            for line in metadata_path.read_text(
                encoding="utf-8"
            ).splitlines()
            if line.strip()
        ]

        repaired = 0
        unresolved = []
        ambiguous = []

        for index, row in enumerate(rows):
            if "file_name" not in row:
                raise RuntimeError(
                    f"{split} row {index} tidak memiliki file_name"
                )

            original_rel = row["file_name"]
            expected = split_root / original_rel

            # Jika path metadata memang sudah benar
            if expected.is_file():
                continue

            basename = Path(original_rel).name

            matches = [
                p
                for p in split_root.rglob(basename)
                if p.is_file()
            ]

            if len(matches) == 1:
                real_file = matches[0]

                new_rel = real_file.relative_to(
                    split_root
                ).as_posix()

                print(
                    f"[{split}] FIX:",
                    original_rel,
                    "->",
                    new_rel,
                )

                row["file_name"] = new_rel
                repaired += 1

            elif len(matches) == 0:
                unresolved.append(
                    {
                        "row": index,
                        "file_name": original_rel,
                    }
                )

            else:
                ambiguous.append(
                    {
                        "row": index,
                        "file_name": original_rel,
                        "matches": [
                            str(x) for x in matches
                        ],
                    }
                )

        # Jangan menebak file kalau tidak jelas
        if unresolved:
            raise RuntimeError(
                f"{split}: ada {len(unresolved)} file "
                f"yang sama sekali tidak ditemukan. "
                f"Contoh: {unresolved[:5]}"
            )

        if ambiguous:
            raise RuntimeError(
                f"{split}: ada {len(ambiguous)} nama file "
                f"yang memiliki lebih dari satu kandidat. "
                f"Contoh: {ambiguous[:3]}"
            )

        # Pastikan tidak ada dua metadata menunjuk file yang sama
        resolved_paths = [
            row["file_name"]
            for row in rows
        ]

        if len(resolved_paths) != len(set(resolved_paths)):
            raise RuntimeError(
                f"{split}: duplicate resolved file path terdeteksi."
            )

        # Tulis metadata runtime yang sudah diperbaiki
        with metadata_path.open(
            "w",
            encoding="utf-8"
        ) as f:
            for row in rows:
                f.write(
                    json.dumps(
                        row,
                        ensure_ascii=False
                    )
                    + "\n"
                )

        # Validasi semua file benar-benar tersedia
        failures = []

        for index, row in enumerate(rows):
            image_path = (
                split_root / row["file_name"]
            )

            if not image_path.is_file():
                failures.append(
                    {
                        "row": index,
                        "file": str(image_path),
                        "reason": "missing",
                    }
                )
                continue

            try:
                with Image.open(image_path) as image:
                    image.verify()

            except Exception as exc:
                failures.append(
                    {
                        "row": index,
                        "file": str(image_path),
                        "reason": repr(exc),
                    }
                )

        if failures:
            raise RuntimeError(
                f"{split}: validasi gambar gagal. "
                f"Contoh: {failures[:5]}"
            )

        report[split] = {
            "metadata_rows": len(rows),
            "repaired_paths": repaired,
            "unresolved": 0,
            "ambiguous": 0,
            "image_validation_failures": 0,
            "status": "valid",
        }

        print()
        print("====================================")
        print("SPLIT:", split)
        print("Metadata:", len(rows))
        print("Path diperbaiki:", repaired)
        print("Unresolved: 0")
        print("Ambiguous: 0")
        print("Image validation: PASS")
        print("====================================")
        print()

    total = (
        report["train"]["metadata_rows"]
        + report["validation"]["metadata_rows"]
    )

    if report["train"]["metadata_rows"] != 312:
        raise RuntimeError(
            "Jumlah train berubah dari expected 312."
        )

    if report["validation"]["metadata_rows"] != 77:
        raise RuntimeError(
            "Jumlah validation berubah dari expected 77."
        )

    if total != 389:
        raise RuntimeError(
            f"Total dataset berubah: {total}, expected 389."
        )

    print("RUNTIME METADATA PATH VALIDATION: PASS")
    print("Train:", report["train"]["metadata_rows"])
    print(
        "Validation:",
        report["validation"]["metadata_rows"]
    )
    print("Total:", total)

    return report


print("Metadata path repair helper: READY")

In [ ]:
from pathlib import Path
import json
import shutil
import hashlib

SOURCE_ROOT = Path(
    "/content/taniadaptif_dataset/multicrop_v1"
)

STAGING_ROOT = Path(
    "/content/taniadaptif_imagefolder"
)

print("Source :", SOURCE_ROOT)
print("Staging:", STAGING_ROOT)

# Buat ulang staging runtime.
# Ini TIDAK mengubah dataset asli.
if STAGING_ROOT.exists():
    shutil.rmtree(STAGING_ROOT)

STAGING_ROOT.mkdir(parents=True, exist_ok=True)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def build_flat_split(split):
    source_split = SOURCE_ROOT / split
    source_metadata = source_split / "metadata.jsonl"

    target_split = STAGING_ROOT / split
    target_split.mkdir(parents=True, exist_ok=True)

    rows = [
        json.loads(line)
        for line in source_metadata.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

    new_rows = []
    source_hashes = {}
    failures = []

    for idx, row in enumerate(rows):
        original_name = row["file_name"]

        # 1. Coba path metadata langsung.
        candidate = source_split / original_name

        if candidate.is_file():
            source_image = candidate

        else:
            # 2. Cari recursive berdasarkan basename.
            basename = Path(original_name).name

            matches = [
                p
                for p in source_split.rglob(basename)
                if p.is_file()
            ]

            if len(matches) == 1:
                source_image = matches[0]

            elif len(matches) == 0:
                failures.append(
                    {
                        "row": idx,
                        "file_name": original_name,
                        "reason": "not_found",
                    }
                )
                continue

            else:
                failures.append(
                    {
                        "row": idx,
                        "file_name": original_name,
                        "reason": "ambiguous",
                        "matches": [
                            str(x) for x in matches
                        ],
                    }
                )
                continue

        extension = source_image.suffix.lower()

        # Nama file baru dibuat unik dan deterministik.
        target_name = (
            f"{split}_{idx:04d}_"
            f"{source_image.stem}{extension}"
        )

        target_image = target_split / target_name

        shutil.copy2(
            source_image,
            target_image
        )

        # Verifikasi byte hasil copy.
        src_hash = sha256_file(source_image)
        dst_hash = sha256_file(target_image)

        if src_hash != dst_hash:
            raise RuntimeError(
                f"Hash berubah saat copy: {source_image}"
            )

        source_hashes[str(source_image)] = src_hash

        new_row = dict(row)

        # ImageFolder sekarang hanya perlu membuka:
        # target_split / target_name
        new_row["file_name"] = target_name

        new_rows.append(new_row)

    if failures:
        raise RuntimeError(
            f"{split}: terdapat masalah file: "
            f"{failures[:10]}"
        )

    metadata_target = target_split / "metadata.jsonl"

    with metadata_target.open(
        "w",
        encoding="utf-8"
    ) as f:
        for row in new_rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                )
                + "\n"
            )

    print()
    print("==============================")
    print("STAGING SPLIT:", split)
    print("Rows:", len(new_rows))
    print("Images:", len(list(target_split.glob('*.*'))) - 1)
    print("Metadata:", metadata_target)
    print("==============================")

    return len(new_rows)


train_count = build_flat_split("train")
validation_count = build_flat_split("validation")

assert train_count == 312, train_count
assert validation_count == 77, validation_count

print()
print("IMAGEFOLDER STAGING CREATED")
print("Train:", train_count)
print("Validation:", validation_count)

In [ ]:
from datasets import load_dataset
from pathlib import Path

TRAIN_STAGING = Path(
    "/content/taniadaptif_imagefolder/train"
)

VALID_STAGING = Path(
    "/content/taniadaptif_imagefolder/validation"
)

print("Menguji TRAIN dengan ImageFolder...")

train_probe = load_dataset(
    "imagefolder",
    data_dir=str(TRAIN_STAGING),
    split="train",
)

print("Train rows:", len(train_probe))

for i in range(len(train_probe)):
    img = train_probe[i]["image"]
    _ = img.size

print("TRAIN IMAGEFOLDER DECODER: PASS")


print()
print("Menguji VALIDATION dengan ImageFolder...")

validation_probe = load_dataset(
    "imagefolder",
    data_dir=str(VALID_STAGING),
    split="validation",
)

print("Validation rows:", len(validation_probe))

for i in range(len(validation_probe)):
    img = validation_probe[i]["image"]
    _ = img.size

print("VALIDATION IMAGEFOLDER DECODER: PASS")

assert len(train_probe) == 312
assert len(validation_probe) == 77

print()
print("====================================")
print("FULL IMAGEFOLDER VALIDATION: PASS")
print("312 train + 77 validation = 389")
print("====================================")

In [ ]:
from pathlib import Path

SMOKE_DIR = Path(
    "/content/drive/MyDrive/skripsi_lora/outputs/smoke_test"
)

SMOKE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Smoke output:", SMOKE_DIR)
print("Training data:", "/content/taniadaptif_imagefolder/train")
print("READY FOR SMOKE TEST")

In [ ]:
!accelerate launch \
  --config_file /content/accelerate_t4.yaml \
  /content/diffusers/examples/text_to_image/train_text_to_image_lora.py \
  --pretrained_model_name_or_path stable-diffusion-v1-5/stable-diffusion-v1-5 \
  --train_data_dir /content/taniadaptif_imagefolder/train \
  --caption_column text \
  --resolution 512 \
  --random_flip \
  --train_batch_size 1 \
  --gradient_accumulation_steps 4 \
  --max_train_steps 100 \
  --checkpointing_steps 50 \
  --checkpoints_total_limit 2 \
  --learning_rate 0.0001 \
  --lr_scheduler constant \
  --lr_warmup_steps 0 \
  --seed 42 \
  --rank 8 \
  --dataloader_num_workers 0 \
  --output_dir /content/drive/MyDrive/skripsi_lora/outputs/smoke_test \
  --report_to tensorboard \
  --logging_dir tensorboard \
  --mixed_precision fp16 \
  --gradient_checkpointing \
  --use_8bit_adam

In [ ]:
from pathlib import Path
from PIL import Image
import json
import shutil
import hashlib

SOURCE_ROOT = Path(
    "/content/taniadaptif_dataset/multicrop_v1"
)

READY_ROOT = Path(
    "/content/taniadaptif_train_ready_v3"
)

print("SOURCE :", SOURCE_ROOT)
print("TARGET :", READY_ROOT)


def sha256_file(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            h.update(chunk)

    return h.hexdigest()


def build_exact_split(split):
    source_split = SOURCE_ROOT / split
    metadata_source = source_split / "metadata.jsonl"

    target_split = READY_ROOT / split

    if target_split.exists():
        shutil.rmtree(target_split)

    target_split.mkdir(
        parents=True,
        exist_ok=True
    )

    rows = [
        json.loads(line)
        for line in metadata_source.read_text(
            encoding="utf-8"
        ).splitlines()
        if line.strip()
    ]

    seen_targets = set()
    unresolved = []
    ambiguous = []

    for index, row in enumerate(rows):

        if "file_name" not in row:
            raise RuntimeError(
                f"{split} row {index}: "
                "tidak memiliki file_name"
            )

        original_rel = row["file_name"]

        rel = Path(original_rel)

        # Keamanan path
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError(
                f"Path metadata tidak aman: {original_rel}"
            )

        # Pertama cari sesuai metadata persis
        source_image = source_split / rel

        if not source_image.is_file():

            basename = rel.name

            matches = [
                p
                for p in source_split.rglob(basename)
                if p.is_file()
            ]

            if len(matches) == 1:
                source_image = matches[0]

            elif len(matches) == 0:
                unresolved.append(
                    {
                        "row": index,
                        "file_name": original_rel
                    }
                )
                continue

            else:
                ambiguous.append(
                    {
                        "row": index,
                        "file_name": original_rel,
                        "matches": [
                            str(x)
                            for x in matches
                        ]
                    }
                )
                continue

        # PENTING:
        # file target memakai path PERSIS
        # seperti yang diminta metadata.
        target_image = target_split / rel

        target_image.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        target_key = target_image.as_posix()

        if target_key in seen_targets:
            raise RuntimeError(
                "Metadata menunjuk dua kali ke target yang sama: "
                f"{target_image}"
            )

        seen_targets.add(target_key)

        shutil.copy2(
            source_image,
            target_image
        )

        # Pastikan copy identik
        if sha256_file(source_image) != sha256_file(target_image):
            raise RuntimeError(
                f"Hash berubah saat copy: {source_image}"
            )

    if unresolved:
        raise RuntimeError(
            f"{split}: file tidak ditemukan: "
            f"{unresolved[:5]}"
        )

    if ambiguous:
        raise RuntimeError(
            f"{split}: basename ambigu: "
            f"{ambiguous[:5]}"
        )

    # Metadata ditulis TANPA mengubah file_name
    metadata_target = target_split / "metadata.jsonl"

    with metadata_target.open(
        "w",
        encoding="utf-8"
    ) as f:

        for row in rows:
            f.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                )
                + "\n"
            )

    # Validasi persis terhadap metadata
    failures = []

    for index, row in enumerate(rows):

        image_path = (
            target_split
            / row["file_name"]
        )

        if not image_path.is_file():
            failures.append(
                (
                    index,
                    row["file_name"],
                    "missing"
                )
            )
            continue

        try:
            with Image.open(image_path) as im:
                im.verify()

        except Exception as exc:
            failures.append(
                (
                    index,
                    row["file_name"],
                    repr(exc)
                )
            )

    if failures:
        raise RuntimeError(
            f"{split}: validation gagal: "
            f"{failures[:10]}"
        )

    print()
    print("============================")
    print("SPLIT:", split)
    print("Rows:", len(rows))
    print("Expected images:", len(rows))
    print("Missing: 0")
    print("Ambiguous: 0")
    print("PIL validation: PASS")
    print("============================")

    return len(rows)


if READY_ROOT.exists():
    shutil.rmtree(READY_ROOT)

READY_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

train_count = build_exact_split("train")
val_count = build_exact_split("validation")

assert train_count == 312
assert val_count == 77

print()
print("======================================")
print("EXACT IMAGEFOLDER STAGING: PASS")
print("Train:", train_count)
print("Validation:", val_count)
print("Total:", train_count + val_count)
print("======================================")

In [ ]:
from pathlib import Path

p = Path(
    "/content/taniadaptif_train_ready_v3/"
    "train/IMG_20240923_181506.jpg"
)

print("Path:", p)
print("Exists:", p.exists())

if p.exists():
    print("Size:", p.stat().st_size, "bytes")

In [ ]:
try:
    record_stage("preparation", "running")
    execute_notebook(PREP_NOTEBOOK, "preparation")
    metadata_report = repair_runtime_metadata_paths()
    record_stage("metadata_path_repair", "completed", metadata_report)
    training_config, evaluation_config = assert_final_config()

    smoke_config = training_config["smoke_test"]
    main_config = training_config["output"]
    SMOKE_DIR = Path(smoke_config["google_drive_directory"])
    MAIN_DIR = Path(main_config["google_drive_directory"])
    SMOKE_LORA = smoke_config["lora_filename"]
    MAIN_LORA = main_config["lora_filename"]
    record_stage(
        "configuration_validation",
        "completed",
        {
            "smoke_lora_filename": SMOKE_LORA,
            "main_lora_filename": MAIN_LORA,
        },
    )

    smoke_report = inspect_training_run(SMOKE_DIR, 100, SMOKE_LORA)
    if smoke_report["valid"]:
        record_stage("smoke_training", "skipped_existing_valid_run")
    else:
        smoke_status = (
            "repairing_evidence_without_retraining"
            if smoke_report["lora_valid"]
            else "running"
        )
        record_stage(
            "smoke_training",
            smoke_status,
            {
                "resume": "latest",
                "canonical_lora_valid": smoke_report["lora_valid"],
                "validation_errors": smoke_report["errors"],
            },
        )
        execute_notebook(
            TRAIN_NOTEBOOK,
            "smoke_training",
            environment={
                "TANIADAPTIF_RUN_MODE": "smoke",
                "TANIADAPTIF_RESUME_FROM_CHECKPOINT": "latest",
            },
        )
    smoke_report = require_training_run(
        SMOKE_DIR, 100, SMOKE_LORA, "orchestrator_smoke_validation.json"
    )
    record_stage("smoke_validation", "completed", smoke_report)

    main_report = inspect_training_run(MAIN_DIR, 1600, MAIN_LORA)
    if main_report["valid"]:
        record_stage("main_training", "skipped_existing_valid_run")
    else:
        main_status = (
            "repairing_evidence_without_retraining"
            if main_report["lora_valid"]
            else "running"
        )
        record_stage(
            "main_training",
            main_status,
            {
                "resume": "latest",
                "canonical_lora_valid": main_report["lora_valid"],
                "validation_errors": main_report["errors"],
            },
        )
        execute_notebook(
            TRAIN_NOTEBOOK,
            "main_training",
            environment={
                "TANIADAPTIF_RUN_MODE": "main",
                "TANIADAPTIF_RESUME_FROM_CHECKPOINT": "latest",
            },
        )
    main_report = require_training_run(
        MAIN_DIR, 1600, MAIN_LORA, "orchestrator_main_validation.json"
    )
    record_stage("main_validation", "completed", main_report)

    evaluation_report = inspect_evaluation(EVALUATION_DIR)
    if evaluation_report["valid"]:
        record_stage("automatic_evaluation", "skipped_existing_valid_run")
    else:
        record_stage("automatic_evaluation", "running")
        execute_notebook(EVAL_NOTEBOOK, "automatic_evaluation")
    evaluation_report = require_evaluation(EVALUATION_DIR)
    record_stage("evaluation_validation", "completed", evaluation_report)

    pipeline_state["result"] = {
        "status": "success",
        "dataset_sha256": EXPECTED_SHA256,
        "smoke": smoke_report,
        "main": main_report,
        "evaluation": evaluation_report,
    }
    record_stage("complete_pipeline", "success", pipeline_state["result"])
except BaseException as exc:
    record_stage(
        "complete_pipeline",
        "failed",
        {"error_type": type(exc).__name__, "message": str(exc)},
    )
    raise

print("Complete pipeline succeeded.")
print("State report:", PIPELINE_STATE_PATH)
print("Main LoRA:", main_report["lora_path"])
print("Evaluation output:", EVALUATION_DIR)


## Batas klaim

Status sukses hanya sah jika cell terakhir selesai tanpa exception dan
`complete_pipeline_state.json` berstatus `success`. Metrik evaluasi otomatis
adalah proxy; template penilaian manusia tetap kosong sampai direview.
